# DepthWizard - 1. prepare the training data

Builds the training tiles for **HeightNet** and saves them as this notebook's
output, so the training notebook can attach them as a dataset.

* **GAMUS** (Hugging Face `earthflow/GAMUS`): US cities, 0.33 m, LiDAR height
  above ground + land-cover labels. Downloaded one tile at a time, converted,
  and deleted, so the disk never holds the 80 GB original.
* **AHN4 (Netherlands) + swisstopo (Switzerland)**: forest, farmland, hills,
  mountains and European towns - the landscapes GAMUS does not have. Tiles
  near any DepthWizard benchmark scene are **excluded**, so the benchmark
  stays a fair test.

**Settings (right-hand panel):** Accelerator **None** (CPU is enough),
Internet **On**. Then **Save Version -> Save & Run All (Commit)** and close
the tab - it runs in the background (roughly 2-4 hours).

**When it has finished:** open the version -> **Output** -> **New Dataset**,
name it `depthwizard-tiles`. That dataset is the input of notebook 2.

In [ ]:
GAMUS_LIMIT = 0          # tiles per split; 0 = all of GAMUS. 30 = a 5-minute trial run
LIDAR_PER_REGION = 35    # tiles per region (31 regions); 5 = a quick trial
OUT = "/kaggle/working/data"

In [ ]:
!pip install -q h5py rasterio huggingface_hub

In [ ]:
# Run a command, stream its output, and STOP the notebook if it fails
# (a plain "!command" carries on after an error).
import subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
# Optional but recommended: a free Hugging Face token lifts the download rate
# limit (GAMUS is ~20,000 files). Add it under Add-ons -> Secrets as HF_TOKEN.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("no HF_TOKEN secret - downloading anonymously (slower if rate-limited)")

In [ ]:
import os
for d in ('dw/mathsandml/benchmark', 'dw/training'):
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile dw/mathsandml/heightnet.py
"""
heightnet.py - DepthWizard's TRAINED height model.
=============================================================================

One RGB image in, METRES ABOVE GROUND out. No scale anchor needed.

The zero-shot path in inference.py uses Depth-Anything-V2 exactly as it was
released: it can only RANK heights, so every absolute run needs one real
measurement (shadows, control points, a landmark height or GHSL) to turn the
ranking into metres. HeightNet is the same backbone after it has been shown
thousands of LiDAR-measured scenes from above, so it answers in metres itself.

WHAT IS IN THE NETWORK
----------------------
  Depth-Anything-V2 encoder + DPT neck   pretrained, fine-tuned. Keeps the
                                         geometry the model already knows.
  GSD conditioning (FiLM)                the model is TOLD how many metres one
                                         pixel covers. The same 10 m block is
                                         30 px wide at 0.33 m and 10 px wide
                                         at 1 m; without this it has to guess.
  semantic head (7 GAMUS classes)        ground / low vegetation / building /
                                         water / road / tree. Trained next to
                                         height because the two are strongly
                                         tied (GAMUS paper, Xiong et al. 2023),
                                         and its output FEEDS the height head.
  relative-depth output (engine A)       the pretrained Depth-Anything head,
                                         fine-tuned with a scale-and-shift-
                                         invariant loss: relative depth made
                                         for top-down images. Goes through the
                                         pipeline's calibration module
                                         (DEM + GCPs / shadows / scene prior)
                                         exactly like the zero-shot model does.
  adaptive-bin height head (engine B)    height is predicted as a probability
                                         over 64 height bins, log-spaced so the
                                         ground and low roofs get fine bins and
                                         skyscrapers coarse ones. A global
                                         token re-spaces the bins per image
                                         (idea from AdaBins / HTC-DC Net).
                                         Expected value = height, spread of
                                         the distribution = uncertainty.

WHY BINS AND NOT PLAIN REGRESSION: height maps are long-tailed - most pixels
are ground (0-2 m), few are tall. A plain regressor learns to predict "low"
everywhere and under-shoots tall buildings, which is exactly the error the
zero-shot pipeline shows (towers above 30 m come out short). Classification
over log-spaced bins plus the long-tail loss weighting in training/losses.py
is the standard cure (HTC-DC Net, Chen et al. 2023).

The checkpoint is self-contained: the architecture config travels inside it,
so loading needs no internet and no Hugging Face download.
"""

from __future__ import annotations

import hashlib
import math
import os

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:          # the pipeline imports this only to ask "is a model there?"
    torch = None

FORMAT = "depthwizard-heightnet-v1"
VARIANTS = {
    "small": "depth-anything/Depth-Anything-V2-Small-hf",   # Apache-2.0
    "base": "depth-anything/Depth-Anything-V2-Base-hf",     # CC-BY-NC-4.0
    "large": "depth-anything/Depth-Anything-V2-Large-hf",   # CC-BY-NC-4.0
}
# GAMUS label order. 0 ("others") is treated as unlabelled during training.
CLASSES = ("others", "ground", "low_vegetation", "building", "water", "road", "tree")
IGNORE_INDEX = 255

H0 = 2.0            # metres; the log-spacing knee: bins are ~linear below, ~log above
H_MAX = 350.0       # tallest height the bins cover (NYC is in GAMUS)
N_BINS = 64
GSD_MIN, GSD_MAX = 0.30, 1.20     # metres per pixel the model was trained across
GSD_DEFAULT = 0.5                 # assumed when an image carries no georeference
PATCH = 14
TILE = 518                        # 37 x 14, Depth-Anything's native grid
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


# ----------------------------------------------------------------------------
# where is the model?  (no torch needed for this part)
# ----------------------------------------------------------------------------
def default_checkpoint():
    """Path the pipeline looks at: $DEPTHWIZARD_HEIGHTNET, else models/heightnet.pt."""
    p = os.environ.get("DEPTHWIZARD_HEIGHTNET", "").strip()
    return p or os.path.join(_REPO_ROOT, "models", "heightnet.pt")


def _is_unzipped_checkpoint(p):
    """A .pt file is a zip archive, and Safari ('open safe files after
    downloading') unpacks it into a FOLDER of the same name. That folder still
    holds the whole model - data.pkl plus data/<n> tensors - so it is accepted
    as-is rather than failing with a confusing error."""
    return os.path.isdir(p) and os.path.isfile(os.path.join(p, "data.pkl"))


def available(path=None):
    """The checkpoint path if a trained model can be used here, else None."""
    p = path or default_checkpoint()
    ok = os.path.isfile(p) or _is_unzipped_checkpoint(p)
    return p if (torch is not None and ok) else None


def file_digest(path, n=1 << 20):
    """Short content hash - part of the prediction cache key."""
    h = hashlib.sha1()
    if os.path.isdir(path):
        with open(os.path.join(path, "data.pkl"), "rb") as f:
            h.update(f.read(n))
        for root, _, names in sorted(os.walk(path)):
            for nm in sorted(names):
                h.update(f"{nm}:{os.path.getsize(os.path.join(root, nm))}".encode())
        return h.hexdigest()[:16]
    with open(path, "rb") as f:
        h.update(f.read(n))
        f.seek(0, 2)
        h.update(str(f.tell()).encode())
    return h.hexdigest()[:16]


def _rezip(folder):
    """The unzipped folder back into the in-memory zip torch.load expects."""
    import io
    import zipfile
    buf = io.BytesIO()
    names = []
    for root, _, files in os.walk(folder):
        for nm in files:
            names.append(os.path.relpath(os.path.join(root, nm), folder))
    names.sort(key=lambda f: (f != "data.pkl", not f.startswith("data" + os.sep), f))
    with zipfile.ZipFile(buf, "w", compression=zipfile.ZIP_STORED) as z:
        for f in names:
            zi = zipfile.ZipInfo("archive/" + f.replace(os.sep, "/"),
                                 date_time=(1980, 1, 1, 0, 0, 0))
            with open(os.path.join(folder, f), "rb") as fh:
                z.writestr(zi, fh.read())
    buf.seek(0)
    return buf


def u_of(h):
    """Height -> the log-like space the bins are uniform in."""
    return torch.log1p(h.clamp(min=0) / H0)


# ----------------------------------------------------------------------------
# COMBINING THE TWO ENGINES  (numpy only - used by the pipeline and evaluate.py)
# ----------------------------------------------------------------------------
#   A  relative depth -> calibration module -> metres   (the problem statement's
#      own workflow: relative depth mapped to absolute height)
#   B  metres directly
# In the hybrid, B plays two parts inside A's workflow:
#   1. the SCENE-LEVEL PRIOR that sets A's scale when no measured anchor
#      (GCPs, shadows) exists - it replaces the 100 m, 2018 GHSL prior;
#   2. a second opinion, fused per pixel by uncertainty.

def robust_affine(src, ref, iters=4, max_n=200_000):
    """Robust ref ~ a*src + b (Cauchy weights), on a subsample for speed."""
    x = np.asarray(src, np.float64).ravel()
    y = np.asarray(ref, np.float64).ravel()
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if x.size > max_n:
        idx = np.random.default_rng(0).choice(x.size, max_n, replace=False)
        x, y = x[idx], y[idx]
    if x.size < 32 or x.std() < 1e-9:
        return 1.0, 0.0
    A = np.c_[x, np.ones_like(x)]
    w = np.ones_like(x)
    coef = np.array([1.0, 0.0])
    for _ in range(iters):
        coef, *_ = np.linalg.lstsq(A * w[:, None], y * w, rcond=None)
        if not np.all(np.isfinite(coef)):
            return 1.0, 0.0
        r = y - A @ coef
        sc = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-9
        w = 1.0 / np.sqrt(1 + (r / (2 * sc)) ** 2)
    a, b = float(coef[0]), float(coef[1])
    return (a, b) if a > 0 else (1.0, 0.0)


def ground_envelope(p, px_size_m, window_m=80.0, pct=2.0, step_m=2.0, max_cells=400):
    """The ground under a relative surface: an 80 m robust opening. Same
    algorithm as inference.ground_envelope, kept here so evaluate.py can run
    without the full pipeline."""
    from scipy.ndimage import gaussian_filter, maximum_filter, percentile_filter, zoom
    a = np.asarray(p, np.float64)
    H, W = a.shape
    px = max(float(px_size_m or 1.0), 1e-6)
    step = max(step_m, px, max(H, W) * px / max_cells)
    f = max(1, int(round(step / px)))
    Hc, Wc = max(1, H // f), max(1, W // f)
    ds = a[:Hc * f, :Wc * f].reshape(Hc, f, Wc, f).mean(axis=(1, 3))
    k = max(3, int(round(window_m / (px * f))) | 1)
    env = percentile_filter(ds, pct, size=k, mode="nearest")
    env = maximum_filter(env, size=k, mode="nearest")
    env = gaussian_filter(env, k / 4.0)
    up = zoom(env, (H / env.shape[0], W / env.shape[1]), order=1)
    if up.shape != (H, W):
        up = np.pad(up, ((0, max(0, H - up.shape[0])), (0, max(0, W - up.shape[1]))),
                    mode="edge")[:H, :W]
    return up


def relative_informative(rel, min_std=1e-4, min_cv=1e-3):
    """False when the relative output carries no structure (a broken or
    untrained head): calibrating it would only amplify noise."""
    r = np.asarray(rel, np.float64)
    r = r[np.isfinite(r)]
    if r.size < 100:
        return False
    sd = float(r.std())
    return sd > min_std and sd > min_cv * abs(float(r.mean()))


LEARNED_PRIOR_PCTS = (90.0, 95.0, 98.0, 99.0, 99.5)


def learned_scale(detail, metric, valid=None, min_height_m=1.0):
    """Metres per relative unit, from B: match the upper height distribution of
    A's detail to B's metres (quantile matching - no pixel-to-pixel pairing, so
    a slightly misplaced edge cannot bias it). None when the scene has nothing
    standing on the ground to match."""
    d = np.asarray(detail, np.float64)
    m = np.asarray(metric, np.float64)
    ok = np.isfinite(d) & np.isfinite(m)
    if valid is not None:
        ok &= valid
    if ok.sum() < 100:
        return None
    d, m = d[ok], m[ok]
    ratios = []
    for q in LEARNED_PRIOR_PCTS:
        dq, mq = np.percentile(d, q), np.percentile(m, q)
        if dq > 1e-9 and mq > min_height_m:
            ratios.append(mq / dq)
    return float(np.median(ratios)) if ratios else None


STACK_BANDS = (0.0, 2.0, 10.0, 30.0)     # metres, on B's own prediction


def stack_weights_for(b_h, fusion):
    """Per-pixel weight on A from the stacked calibration: one weight per band
    of B's predicted height, learned on held-out tiles."""
    edges = list(fusion.get("bands", STACK_BANDS))
    w = np.asarray(fusion["w_a"], np.float64)
    idx = np.clip(np.searchsorted(edges, np.nan_to_num(b_h), side="right") - 1,
                  0, len(w) - 1)
    return w[idx]


def fuse(a_h, a_sig, b_h, b_sig, fusion=None, floor_m=0.3):
    """Blend A (calibrated) and B per pixel. Returns (height, sigma, weight_of_B).

    fusion["mode"] (written by evaluate.py after measuring on held-out tiles):
      "ivw"      inverse-variance, each engine's own sigma scaled by its measured
                 honesty factor (c_a, c_b). The default without a calibration.
      "stacked"  a fixed weight per band of B's predicted height (w_a), fitted by
                 least squares on held-out tiles. Chosen by evaluate.py when it
                 beats ivw there - e.g. when A's sigma misses its biggest error,
                 under-scaled tall buildings, which the rotation spread cannot see.
    """
    f = fusion or {}
    ca, cb = float(f.get("c_a", 1.0)), float(f.get("c_b", 1.0))
    va = (ca * np.maximum(np.nan_to_num(a_sig, nan=1e3), floor_m)) ** 2
    vb = (cb * np.maximum(np.nan_to_num(b_sig, nan=1e3), floor_m)) ** 2
    if f.get("mode") == "stacked" and f.get("w_a") is not None:
        wa = stack_weights_for(b_h, f)
        wb = 1.0 - wa
        h = wa * a_h + wb * b_h
        return h, np.sqrt(wa ** 2 * va + wb ** 2 * vb), wb
    wb = va / (va + vb)
    h = (1 - wb) * a_h + wb * b_h
    return h, np.sqrt(va * vb / (va + vb)), wb


def combine(pred, px_size_m, fusion=None, alpha=None, envelope=None):
    """A, B and A+B from one HeightPredictor.predict() result (heights above
    ground, metres). `alpha` = A's scale if a measured anchor supplied one;
    otherwise B sets it (the learned prior). `envelope` lets the pipeline pass
    its own ground model."""
    rel = np.asarray(pred["rel"], np.float64)
    if not relative_informative(rel):
        return dict(alpha=None, alpha_source="none: the relative output is flat",
                    a=None, a_sigma=None, fused=pred["height"], fused_sigma=pred["sigma"],
                    w_b=np.ones_like(pred["height"]), detail=np.zeros_like(rel))
    env = envelope(rel) if envelope else ground_envelope(rel, px_size_m)
    detail = rel - env
    src = "given"
    if alpha is None:
        alpha = learned_scale(detail, pred["height"])
        src = "learned prior (HeightNet metric head)"
    if alpha is None:
        return dict(alpha=None, alpha_source="none: nothing above ground to match",
                    a=None, a_sigma=None, fused=pred["height"], fused_sigma=pred["sigma"],
                    w_b=np.ones_like(pred["height"]), detail=detail)
    a_h = np.maximum(alpha * detail, 0.0)
    a_sig = alpha * np.asarray(pred.get("rel_sigma", np.zeros_like(rel)), np.float64)
    h, sig, wb = fuse(a_h, a_sig, pred["height"], pred["sigma"], fusion)
    return dict(alpha=float(alpha), alpha_source=src, a=a_h, a_sigma=a_sig,
                fused=np.maximum(h, 0.0), fused_sigma=sig, w_b=wb, detail=detail)


if torch is not None:

    class _ConvGN(nn.Sequential):
        def __init__(self, cin, cout, k=3):
            super().__init__(nn.Conv2d(cin, cout, k, padding=k // 2, bias=False),
                             nn.GroupNorm(min(8, cout), cout), nn.GELU())

    class GSDEmbed(nn.Module):
        """metres-per-pixel -> a vector. Fourier features of log2(gsd / 0.5 m)."""

        def __init__(self, dim=128, n_freq=6):
            super().__init__()
            self.register_buffer("freqs", (2.0 ** torch.arange(n_freq)) * math.pi / 4,
                                 persistent=False)
            self.mlp = nn.Sequential(nn.Linear(1 + 2 * n_freq, dim), nn.GELU(),
                                     nn.Linear(dim, dim))

        def forward(self, gsd):
            x = torch.log2(gsd.float().clamp(0.05, 20.0) / GSD_DEFAULT)[:, None]
            z = x * self.freqs[None]
            return self.mlp(torch.cat([x, z.sin(), z.cos()], -1))

    class HeightNet(nn.Module):
        """Depth-Anything-V2 body + GSD FiLM + semantic head + adaptive-bin height head."""

        def __init__(self, da, n_bins=N_BINS, h_max=H_MAX, n_classes=len(CLASSES),
                     emb_dim=128):
            super().__init__()
            self.da = da
            cfg = da.config
            fch = int(cfg.fusion_hidden_size)
            hid = int(cfg.backbone_config.hidden_size)
            hh = int(cfg.head_hidden_size)
            self.n_bins, self.h_max = int(n_bins), float(h_max)
            self.n_classes, self.emb_dim = int(n_classes), int(emb_dim)

            self.gsd_embed = GSDEmbed(emb_dim)
            # FiLM on the finest fused feature. Zero-initialised: at step 0 the
            # network is exactly the pretrained one, the GSD signal fades in.
            self.film = nn.Linear(emb_dim, 2 * fch)
            nn.init.zeros_(self.film.weight)
            nn.init.zeros_(self.film.bias)

            cin = fch // 2 + hh + 1          # conv1 out + conv2 out + relative depth
            self.seg_head = nn.Sequential(_ConvGN(cin, 64), _ConvGN(64, 64),
                                          nn.Conv2d(64, n_classes, 1))
            self.height_trunk = nn.Sequential(_ConvGN(cin + n_classes, 96),
                                              _ConvGN(96, 96))
            self.height_logits = nn.Conv2d(96, n_bins, 1)

            edges_u = torch.linspace(0.0, math.log1p(h_max / H0), n_bins + 1)
            edges_h = H0 * torch.expm1(edges_u)
            w = edges_h[1:] - edges_h[:-1]
            self.register_buffer("base_w", w / w.sum())
            # start the height distribution near the ground rather than at the
            # mean bin (~60 m): a long-tailed prior on the logits' bias
            cu = 0.5 * (edges_u[1:] + edges_u[:-1])
            with torch.no_grad():
                self.height_logits.bias.copy_(-1.5 * cu)

            self.bin_mlp = nn.Sequential(nn.Linear(hid + fch + emb_dim, 256), nn.GELU(),
                                         nn.Linear(256, n_bins))
            nn.init.zeros_(self.bin_mlp[-1].weight)
            nn.init.zeros_(self.bin_mlp[-1].bias)

        # -- bins ----------------------------------------------------------
        def bin_centers(self, glob):
            """Per-image bin centres (B, K) in metres. Zero MLP output = the base log grid."""
            t = 1.5 * torch.tanh(self.bin_mlp(glob.float()))
            w = self.base_w[None] * t.exp()
            w = w / w.sum(-1, keepdim=True)
            edges = self.h_max * torch.cumsum(w, -1)
            edges = torch.cat([torch.zeros_like(edges[:, :1]), edges], -1)
            return 0.5 * (edges[:, :-1] + edges[:, 1:])

        # -- forward -------------------------------------------------------
        def forward(self, pixel_values, gsd):
            """pixel_values (B,3,H,W) ImageNet-normalised, H and W multiples of 14.
            gsd (B,) metres per pixel. Outputs are at H/2 x W/2."""
            B, _, H, W = pixel_values.shape
            ph, pw = H // PATCH, W // PATCH
            bb = self.da.backbone(pixel_values)
            fmaps = list(bb.feature_maps)
            cls = fmaps[-1][:, 0]
            fused = self.da.neck(fmaps, ph, pw)
            head = self.da.head
            f = fused[getattr(head, "head_in_index", -1)]

            e = self.gsd_embed(gsd)
            g, b = self.film(e).to(f.dtype).chunk(2, -1)
            f = f * (1 + g[..., None, None]) + b[..., None, None]

            oh, ow = ph * PATCH // 2, pw * PATCH // 2
            x1 = F.interpolate(head.conv1(f), (oh, ow), mode="bilinear", align_corners=True)
            x2 = F.relu(head.conv2(x1))
            # the pretrained relative depth, made scale-free: its units change
            # from image to image, a log turns that into an offset, and
            # removing the mean removes the offset. (Dividing by the std
            # instead explodes the gradient on flat scenes such as water.)
            rel = F.relu(head.conv3(x2))[:, 0]                 # engine A output
            d = torch.log1p(rel.float())[:, None]
            d = (d - d.mean((2, 3), keepdim=True)).to(x2.dtype)
            feat = torch.cat([x1, x2, d], 1)

            seg = self.seg_head(feat)
            t = self.height_trunk(torch.cat([feat, seg.softmax(1)], 1))
            logits = self.height_logits(t)

            glob = torch.cat([cls.float(), f.float().mean((2, 3)), e.float()], -1)
            centers = self.bin_centers(glob)
            prob = logits.float().softmax(1)
            c = centers[:, :, None, None]
            h = (prob * c).sum(1)
            var = (prob * (c - h[:, None]) ** 2).sum(1)
            return dict(height=h, sigma=var.clamp_min(1e-6).sqrt(), logits=logits,
                        centers=centers, seg=seg, rel=rel)

    # ------------------------------------------------------------------------
    # build / save / load
    # ------------------------------------------------------------------------
    def build(variant="small", pretrained=True, da_config=None, **kw):
        """A fresh HeightNet. `pretrained` downloads Depth-Anything-V2 weights (training
        machines only); `da_config` builds the same shape offline with random weights."""
        from transformers import DepthAnythingConfig, DepthAnythingForDepthEstimation
        if da_config is not None:
            cfg = (DepthAnythingConfig.from_dict(dict(da_config))
                   if isinstance(da_config, dict) else da_config)
            da = DepthAnythingForDepthEstimation(cfg)
        else:
            repo = VARIANTS.get(variant, variant)
            if pretrained:
                da = DepthAnythingForDepthEstimation.from_pretrained(repo)
            else:
                da = DepthAnythingForDepthEstimation(DepthAnythingConfig.from_pretrained(repo))
        return HeightNet(da, **kw)

    def tiny_da_config():
        """A few-MB Depth-Anything shape for tests - same code path, random weights."""
        from transformers import DepthAnythingConfig, Dinov2Config
        bb = Dinov2Config(hidden_size=48, num_hidden_layers=4, num_attention_heads=2,
                          intermediate_size=96, image_size=518, patch_size=14,
                          out_indices=[1, 2, 3, 4], reshape_hidden_states=False)
        return DepthAnythingConfig(backbone_config=bb, neck_hidden_sizes=[12, 24, 48, 48],
                                   fusion_hidden_size=32, head_hidden_size=16,
                                   reassemble_hidden_size=48, patch_size=14)

    def _torch_load(path):
        src = _rezip(path) if _is_unzipped_checkpoint(path) else path
        try:
            return torch.load(src, map_location="cpu", weights_only=True)
        except Exception:
            if hasattr(src, "seek"):
                src.seek(0)
            return torch.load(src, map_location="cpu", weights_only=False)

    def set_fusion(path, fusion):
        """Write evaluate.py's fusion calibration into an existing checkpoint."""
        ck = _torch_load(path)
        ck["fusion"] = {k: (v if isinstance(v, (str, list)) else float(v))
                        for k, v in fusion.items()}
        tmp = path + ".tmp"
        torch.save(ck, tmp)
        if os.path.isdir(path):                 # replace a Safari folder with the file
            import shutil
            shutil.rmtree(path)
        os.replace(tmp, path)

    def save_checkpoint(path, model, variant="small", extra=None, half=True):
        m = model.module if hasattr(model, "module") else model
        sd = {k: (v.detach().half() if half and v.is_floating_point() else v.detach()).cpu()
              for k, v in m.state_dict().items()}
        ck = dict(format=FORMAT, variant=variant, da_config=m.da.config.to_dict(),
                  heightnet=dict(n_bins=m.n_bins, h_max=m.h_max, n_classes=m.n_classes,
                                 emb_dim=m.emb_dim),
                  classes=list(CLASSES), gsd_range=[GSD_MIN, GSD_MAX],
                  state_dict=sd, train=extra or {})
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        tmp = path + ".tmp"
        torch.save(ck, tmp)
        os.replace(tmp, path)
        return path

    def load_checkpoint(path, device="cpu"):
        ck = _torch_load(path)
        if not isinstance(ck, dict) or ck.get("format") != FORMAT:
            raise ValueError(f"{path} is not a DepthWizard HeightNet checkpoint")
        model = build(da_config=ck["da_config"], **ck["heightnet"])
        sd = {k: (v.float() if v.is_floating_point() else v) for k, v in ck["state_dict"].items()}
        model.load_state_dict(sd, strict=True)
        return model.eval().to(device), ck

    def pick_device(want="auto"):
        want = (want or "auto").lower()
        if want == "cuda" or (want == "auto" and torch.cuda.is_available()):
            return torch.device("cuda")
        if want == "mps" or (want == "auto" and getattr(torch.backends, "mps", None)
                             and torch.backends.mps.is_available()):
            return torch.device("mps")
        return torch.device("cpu")

    def normalise(rgb_u8):
        """(N,H,W,3) or (H,W,3) uint8 -> float tensor (N,3,H,W), ImageNet stats."""
        a = rgb_u8.astype(np.float32) / 255.0
        a = (a - MEAN) / STD
        if a.ndim == 3:
            a = a[None]
        return torch.from_numpy(np.ascontiguousarray(a.transpose(0, 3, 1, 2)))

    # ------------------------------------------------------------------------
    # inference on a whole scene
    # ------------------------------------------------------------------------
    def _positions(n, tile, stride):
        if n <= tile:
            return [0]
        p = list(range(0, n - tile + 1, stride))
        if p[-1] != n - tile:
            p.append(n - tile)
        return p

    def _window(tile, ov):
        """Blend weights: sin^2 ramp over the overlap, flat in the middle."""
        r = np.ones(tile, np.float32)
        if ov > 0:
            ramp = np.sin(np.linspace(0, np.pi / 2, ov + 2)[1:-1]) ** 2
            r[:ov] = ramp
            r[-ov:] = ramp[::-1]
        return np.maximum(np.outer(r, r), 1e-3)

    class HeightPredictor:
        """Loads a checkpoint once; predict() handles any image size and GSD."""

        def __init__(self, path=None, device="auto", batch=None):
            path = path or default_checkpoint()
            self.path = path
            self.device = pick_device(device)
            self.model, self.ckpt = load_checkpoint(path, "cpu")
            self.model.to(self.device)
            self.digest = file_digest(path)
            self.batch = batch or (4 if self.device.type == "cuda" else 1)
            self.variant = self.ckpt.get("variant", "?")
            self.train_info = self.ckpt.get("train", {})
            self.fusion = dict(self.ckpt.get("fusion") or {})

        def describe(self):
            t = self.train_info or {}
            data = t.get("data") or "LiDAR-labelled aerial scenes"
            return f"HeightNet-{self.variant} ({data})"

        @torch.no_grad()
        def _run(self, tiles, gsd):
            x = normalise(tiles).to(self.device)
            g = torch.full((x.shape[0],), float(gsd), device=self.device)
            if self.device.type == "cuda":
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(x, g)
            else:
                out = self.model(x, g)
            size = tiles.shape[1:3]

            def up(t):
                return F.interpolate(t[:, None].float(), size=size, mode="bilinear",
                                     align_corners=False)[:, 0].cpu().numpy()
            p = F.interpolate(out["seg"].float().softmax(1), size=size, mode="bilinear",
                              align_corners=False)
            return up(out["height"]), up(out["sigma"]), p.cpu().numpy(), up(out["rel"])

        def _tiled(self, img, gsd, overlap=128):
            H, W = img.shape[:2]
            T = TILE
            ph, pw = max(0, T - H), max(0, T - W)
            if ph or pw:
                img = np.pad(img, ((0, ph), (0, pw), (0, 0)), mode="reflect")
            Hp, Wp = img.shape[:2]
            stride = T - overlap
            win = _window(T, overlap)
            K = self.model.n_classes
            acc_h = np.zeros((Hp, Wp), np.float32)
            acc_s = np.zeros((Hp, Wp), np.float32)
            acc_r = np.zeros((Hp, Wp), np.float32)
            acc_p = np.zeros((K, Hp, Wp), np.float32)
            acc_w = np.zeros((Hp, Wp), np.float32)
            boxes = [(y, x) for y in _positions(Hp, T, stride) for x in _positions(Wp, T, stride)]
            for i in range(0, len(boxes), self.batch):
                chunk = boxes[i:i + self.batch]
                tiles = np.stack([img[y:y + T, x:x + T] for y, x in chunk])
                h, s, p, rel = self._run(tiles, gsd)
                for j, (y, x) in enumerate(chunk):
                    # metres need no alignment; the RELATIVE output is only
                    # defined up to scale and shift per tile, so each tile is
                    # matched to what is already stitched in the overlap - the
                    # same tile scale alignment the zero-shot engine uses
                    seen = acc_w[y:y + T, x:x + T] > 1e-6
                    rj = rel[j]
                    if seen.sum() > 256:
                        ref = acc_r[y:y + T, x:x + T][seen] / acc_w[y:y + T, x:x + T][seen]
                        a, b = robust_affine(rj[seen], ref)
                        rj = a * rj + b
                    acc_h[y:y + T, x:x + T] += h[j] * win
                    acc_s[y:y + T, x:x + T] += (s[j] ** 2) * win
                    acc_r[y:y + T, x:x + T] += rj * win
                    acc_p[:, y:y + T, x:x + T] += p[j] * win
                    acc_w[y:y + T, x:x + T] += win
            acc_w = np.maximum(acc_w, 1e-6)
            return ((acc_h / acc_w)[:H, :W], np.sqrt(acc_s / acc_w)[:H, :W],
                    (acc_p / acc_w)[:, :H, :W], (acc_r / acc_w)[:H, :W], len(boxes))

        def predict(self, rgb, gsd_m=None, rotations=4, overlap=128, progress=None):
            """rgb uint8 (H,W,3). gsd_m metres/pixel (None = not georeferenced).

            Returns, on the input grid:
              height      engine B: metres above ground
              sigma       B's 1-sigma uncertainty, metres (bin spread + rotation spread)
              rel         engine A: relative depth (scale and shift undefined)
              rel_sigma   A's spread across rotations, in rel units
              seg         class index per pixel (CLASSES), seg_conf its probability
            """
            import cv2
            H, W = rgb.shape[:2]
            gsd_in = float(gsd_m) if gsd_m else GSD_DEFAULT
            work = min(max(gsd_in, GSD_MIN), GSD_MAX)
            s = gsd_in / work
            img = rgb
            if abs(s - 1.0) > 0.02:
                nw, nh = max(8, int(round(W * s))), max(8, int(round(H * s)))
                img = cv2.resize(rgb, (nw, nh),
                                 interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
            n_rot = int(max(1, min(4, rotations)))
            mean = m2 = s2 = pr = None
            rmean = rm2 = None
            tiles = 0
            for i, k in enumerate(range(n_rot)):
                r = np.ascontiguousarray(np.rot90(img, k))
                h, sg, p, rel, nt = self._tiled(r, work, overlap)
                tiles += nt
                h = np.rot90(h, -k)
                sg = np.rot90(sg, -k)
                p = np.rot90(p, -k, axes=(1, 2))
                rel = np.rot90(rel, -k)
                if mean is None:
                    mean, m2 = h.copy(), np.zeros_like(h)
                    s2, pr = sg ** 2, p.copy()
                    rmean, rm2 = rel.copy(), np.zeros_like(rel)
                else:                                   # Welford across rotations
                    d = h - mean
                    mean += d / (i + 1)
                    m2 += d * (h - mean)
                    s2 += sg ** 2
                    pr += p
                    a, b = robust_affine(rel, rmean)    # onto the first pass's scale
                    rel = a * rel + b
                    d = rel - rmean
                    rmean += d / (i + 1)
                    rm2 += d * (rel - rmean)
                if progress:
                    progress((i + 1) / n_rot)
            s2 /= n_rot
            pr /= n_rot
            sigma = np.sqrt(s2 + m2 / n_rot)
            rsig = np.sqrt(rm2 / n_rot)
            if img is not rgb:
                def back(a):
                    return cv2.resize(np.ascontiguousarray(a), (W, H),
                                      interpolation=cv2.INTER_LINEAR)
                mean, sigma, rmean, rsig = back(mean), back(sigma), back(rmean), back(rsig)
                pr = np.stack([back(c) for c in pr])
            seg = pr.argmax(0).astype(np.uint8)
            return dict(height=np.maximum(mean, 0).astype(np.float32),
                        sigma=sigma.astype(np.float32), seg=seg,
                        seg_conf=pr.max(0).astype(np.float32),
                        rel=rmean.astype(np.float32), rel_sigma=rsig.astype(np.float32),
                        gsd_input=gsd_in, gsd_model=work, rotations=n_rot, tiles=tiles,
                        georeferenced=bool(gsd_m))


# ----------------------------------------------------------------------------
# CLI: python heightnet.py image.tif [--ckpt models/heightnet.pt]
# ----------------------------------------------------------------------------
def _main():
    import argparse
    ap = argparse.ArgumentParser(description="Run the trained height model on one image.")
    ap.add_argument("image")
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--rotations", type=int, default=4)
    ap.add_argument("--out", default=None, help="write height (m, float32) as .npy")
    a = ap.parse_args()
    import sys
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    import inference as I
    rgb, meta = I.load_image(a.image)
    hp = HeightPredictor(a.ckpt)
    r = hp.predict(rgb, meta.get("px_size_m"), rotations=a.rotations)
    h = r["height"]
    print(f"{hp.describe()}  gsd {r['gsd_input']:.2f} m -> model {r['gsd_model']:.2f} m  "
          f"{r['tiles']} tile passes")
    print(f"height above ground: p50 {np.percentile(h, 50):.1f} m  p99 "
          f"{np.percentile(h, 99):.1f} m  max {h.max():.1f} m  "
          f"mean sigma {r['sigma'].mean():.2f} m")
    if a.out:
        np.save(a.out, h)


if __name__ == "__main__":
    _main()

In [ ]:
%%writefile dw/mathsandml/preprocess.py
"""
preprocess.py - condition the image BEFORE the depth model sees it.
=============================================================================

WHY THIS EXISTS. load_image() used to hand the backbone whatever came out of
the file. That is fine for a clean 8-bit aerial crop and quietly wrong for
almost everything a real archive contains:

  * an 11-bit sensor value packed in a 16-bit container, so the whole scene
    occupies the bottom 3% of the range and reads as flat grey
  * a false-colour product whose first three bands are NIR, R, G - the
    backbone was trained on natural RGB, and near-infrared reflectance looks
    nothing like a red channel to a network that has never seen it labelled
    as one. Vegetation goes bright white, and the depth prior breaks
  * haze, which flattens local contrast across the whole frame
  * cloud and its shadow, which are not surfaces but which the model happily
    assigns a depth to, and which then skew every percentile the pipeline
    computes downstream

None of these are exotic. They are what Landsat, Sentinel and most commercial
deliveries actually look like.

WHAT IT DOES NOT DO. It does not sharpen, denoise or otherwise invent
structure. Everything here is a monotonic, per-band intensity remap plus a
mask - the geometry in the image is untouched, because the whole pipeline's
accuracy claim rests on that geometry being the sensor's, not ours.

ORDER MATTERS:
    1. pick the real RGB bands      (before anything reads pixel values)
    2. build the valid mask          (before the stretch, so cloud cannot
                                      skew the percentiles)
    3. per-band percentile stretch   (bit depth -> 0..255)
    4. CLAHE on luminance only       (local contrast, no colour shift)

Returns the conditioned uint8 RGB plus a report, so what was applied ends up
in the job log and in meta.json rather than only in someone's memory.
"""

import numpy as np

# Percentile clip for the stretch. 2/98 rather than min/max: one saturated
# pixel (a roof specular, a hot pixel) would otherwise crush everything else
# into a handful of grey levels.
STRETCH_LO_PCT = 2.0
STRETCH_HI_PCT = 98.0

# Below this, an 8-bit image is using so little of its range that it is worth
# stretching even though it is already uint8. A well-exposed photo spans
# 150-250 levels; 90 is a genuinely flat, hazy frame.
NARROW_RANGE_LEVELS = 90.0

# CLAHE. Deliberately mild. A high clip limit manufactures texture in flat
# regions, and the depth model will read manufactured texture as structure -
# which is the one failure this file must not cause.
CLAHE_CLIP = 2.0
CLAHE_GRID = 8

# Cloud/shadow thresholds, in ROBUST UNITS - multiples of the interquartile
# range away from the median luminance.
#
# Not absolute 0..1 thresholds, and not percentiles either. Both fail the
# same way: a cloud covering 3% of the frame sits inside the 99th percentile,
# so any normalisation anchored on the extremes gets dragged out to meet it,
# and then every ordinary surface looks black by comparison. The median and
# the IQR cannot be moved by anything smaller than a quarter of the image, so
# "much brighter than typical" stays meaningful no matter what the sensor's
# units are or how much cloud is in frame.
CLOUD_K = 2.5              # IQRs above the median ...
CLOUD_SAT = 0.15           # ... and nearly colourless
SHADOW_K = 2.5             # IQRs below the median
MASK_MAX_FRAC = 0.35       # if more than this is masked, distrust the mask


# ---------------------------------------------------------------------------
# 1. band selection
# ---------------------------------------------------------------------------
def pick_rgb_bands(ds):
    """Which band indices are actually Red, Green, Blue? (1-based, rasterio.)

    Taking the first three bands is the common shortcut and it is wrong on
    two very ordinary cases: a 4-band product ordered (B,G,R,NIR), and a
    false-colour aerial ordered (NIR,R,G). Both are widespread.

    rasterio exposes the answer when the file bothers to declare it, via
    colorinterp; we fall back to band descriptions, then to "first three"
    with a flag so the caller can say so in the log rather than silently
    feeding infrared to the backbone.
    """
    from rasterio.enums import ColorInterp

    n = ds.count
    if n == 1:
        return (1, 1, 1), "single band, replicated to grey"

    # (a) declared colour interpretation - the authoritative answer
    try:
        ci = list(ds.colorinterp)
        want = (ColorInterp.red, ColorInterp.green, ColorInterp.blue)
        if all(c in ci for c in want):
            idx = tuple(ci.index(c) + 1 for c in want)
            if idx != (1, 2, 3):
                return idx, f"bands reordered from colorinterp -> R{idx[0]} G{idx[1]} B{idx[2]}"
            return idx, None
    except Exception:
        pass

    # (b) band descriptions, e.g. ('nir', 'red', 'green')
    desc = [(d or "").strip().lower() for d in (ds.descriptions or [])]
    if desc and any(desc):
        def find(*names):
            for i, d in enumerate(desc):
                if any(d == nm or d.startswith(nm) for nm in names):
                    return i + 1
            return None
        r, g, b = find("red", "r"), find("green", "g"), find("blue", "b")
        if r and g and b:
            idx = (r, g, b)
            if idx != (1, 2, 3):
                return idx, f"bands reordered from descriptions -> R{r} G{g} B{b}"
            return idx, None
        # false-colour: NIR present, no blue. Substitute green for blue -
        # a cheap natural-ish proxy. DAv2 cares about structure, and a
        # (R,G,G) stack keeps vegetation from blowing out to white.
        nir = find("nir", "near")
        if nir and r and g and not b:
            return (r, g, g), "false colour (NIR present, no blue) - using (R,G,G) proxy"

    # (c) give up, but say so
    return (1, 2, 3), "band roles undeclared - assuming first three are RGB"


# ---------------------------------------------------------------------------
# 2. valid mask
# ---------------------------------------------------------------------------
def valid_mask(rgb_raw, nodata_mask=None):
    """True where the pixel is a real surface we can trust.

    Cloud has no depth and the model will still give it one. Worse, cloud is
    bright and shadow is dark, so leaving them in skews every percentile the
    pipeline computes - the stretch, the structure-scale estimate, and the
    height reference the metre scale is fitted against.

    rgb_raw : the band stack in its NATIVE units. No normalisation is applied
        first, deliberately: every scheme for mapping the data to 0..1 is
        itself distorted by the outliers we are trying to find.

    Returns (mask, report).
    """
    f = rgb_raw.astype(np.float64)
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    # Saturation is a ratio, so it is already free of the sensor's units.
    mx = np.maximum(np.maximum(r, g), b)
    mn = np.minimum(np.minimum(r, g), b)
    sat = np.where(np.abs(mx) > 1e-9, (mx - mn) / np.maximum(np.abs(mx), 1e-9), 0.0)

    finite = np.isfinite(luma)
    if not finite.any():
        return np.zeros(luma.shape, bool), dict(masked_frac=1.0,
                                                mask_rejected="no finite pixels")
    q25, med, q75 = np.percentile(luma[finite], [25, 50, 75])
    iqr = float(q75 - q25)
    if iqr < 1e-9:
        # a genuinely uniform frame - nothing stands out, so nothing is masked
        return np.ones(luma.shape, bool), dict(cloud_frac=0.0, shadow_frac=0.0,
                                               masked_frac=0.0,
                                               mask_note="flat histogram, no mask applied")

    cloud = (luma > med + CLOUD_K * iqr) & (sat < CLOUD_SAT)
    shadow = luma < med - SHADOW_K * iqr
    bad = (cloud | shadow) & finite
    bad |= ~finite
    if nodata_mask is not None:
        bad |= nodata_mask

    frac = float(bad.mean())
    rep = dict(cloud_frac=float(cloud.mean()),
               shadow_frac=float(shadow.mean()),
               masked_frac=frac)

    # A mask covering a third of the frame is more likely a thresholding
    # failure (a snowfield, a night scene, a desert) than genuine cloud.
    # Fall back to "everything valid" and say why, rather than throwing away
    # the scene.
    if frac > MASK_MAX_FRAC:
        rep["mask_rejected"] = (f"masked {100 * frac:.0f}% of the frame - "
                                f"treating the whole scene as valid instead")
        return np.ones(luma.shape, bool), rep

    return ~bad, rep


# ---------------------------------------------------------------------------
# 3. stretch
# ---------------------------------------------------------------------------
def stretch_per_band(arr, mask=None, lo_pct=STRETCH_LO_PCT, hi_pct=STRETCH_HI_PCT):
    """Percentile stretch EACH band independently, to 0..255 uint8.

    Per band, not joint. Atmospheric scattering is wavelength dependent - blue
    scatters most - so a hazy scene arrives with a colour cast and a joint
    stretch preserves it. Stretching each band to the same percentiles removes
    the cast, which is a crude but effective white balance.

    Percentiles are taken over `mask` only, so a cloud cannot define the
    bright end and a shadow cannot define the dark end.
    """
    out = np.empty(arr.shape[:2] + (3,), np.uint8)
    lohi = []
    for c in range(3):
        band = arr[..., c].astype(np.float64)
        sample = band[mask] if mask is not None else band.ravel()
        sample = sample[np.isfinite(sample)]
        if sample.size < 16:
            sample = band[np.isfinite(band)].ravel()
        if sample.size == 0:
            out[..., c] = 0
            lohi.append((0.0, 0.0))
            continue
        lo, hi = np.percentile(sample, [lo_pct, hi_pct])
        if hi - lo < 1e-9:
            hi = lo + 1e-9
        # NaN survives clip() and casting it to uint8 is undefined - on this
        # machine it lands on 0, on another it could be anything. Send holes
        # to the dark end explicitly.
        scaled = np.clip((band - lo) / (hi - lo), 0, 1)
        out[..., c] = (np.where(np.isfinite(scaled), scaled, 0.0) * 255).astype(np.uint8)
        lohi.append((float(lo), float(hi)))
    return out, lohi


def needs_stretch(arr):
    """Is this image worth stretching? Returns (bool, reason)."""
    if arr.dtype != np.uint8:
        return True, f"{arr.dtype} input - compressing to 8-bit"
    lo, hi = np.percentile(arr, [STRETCH_LO_PCT, STRETCH_HI_PCT])
    span = float(hi - lo)
    if span < 1.0:
        # A uniform frame has nothing to stretch, and stretching it anyway
        # maps every pixel to the dark end - a synthetic all-grey test tile
        # came out black. Leave it exactly as it arrived.
        return False, None
    if span < NARROW_RANGE_LEVELS:
        return True, f"8-bit but only {span:.0f} levels of range - hazy or under-exposed"
    return False, None


# ---------------------------------------------------------------------------
# 4. local contrast
# ---------------------------------------------------------------------------
def clahe_luma(rgb_u8, clip=CLAHE_CLIP, grid=CLAHE_GRID):
    """CLAHE on the L channel of LAB, leaving colour alone.

    Applied per channel in RGB it shifts hue, which then breaks the green
    index buildings.py uses to reject vegetation. Applied to luminance only,
    it recovers local contrast in haze without touching the colour the rest
    of the pipeline reasons about.
    """
    import cv2
    lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
    c = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(grid), int(grid)))
    lab[..., 0] = c.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


# ---------------------------------------------------------------------------
# the whole chain
# ---------------------------------------------------------------------------
def condition(arr, nodata_mask=None, do_stretch=True, do_clahe=True,
              do_mask=True):
    """raw band stack (HxWx3, any dtype) -> (uint8 RGB, mask, report).

    `mask` is True where the pixel is trustworthy. It is returned rather than
    applied, because the right thing to do with a cloud differs by stage: the
    stretch excludes it from its percentiles, the calibrator should exclude it
    from its sample, and the texture drape should keep it so the render still
    looks like the input.
    """
    rep = {}
    a = np.asarray(arr)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, axis=2)

    # The mask runs on the NATIVE values. It uses median/IQR, which need no
    # normalisation and cannot be dragged around by the very pixels they are
    # looking for.
    mask = None
    if do_mask:
        mask, mrep = valid_mask(a, nodata_mask)
        rep.update(mrep)

    want, why = needs_stretch(a)
    if do_stretch and want:
        u8, lohi = stretch_per_band(a, mask)
        rep["stretch"] = why
        rep["stretch_lohi"] = lohi
    else:
        u8 = a.astype(np.uint8) if a.dtype == np.uint8 else \
             np.clip(f01 * 255, 0, 255).astype(np.uint8)

    if do_clahe:
        try:
            u8 = clahe_luma(u8)
            rep["clahe"] = f"clip={CLAHE_CLIP} grid={CLAHE_GRID}"
        except Exception as ex:
            # never let a contrast tweak take the run down
            rep["clahe_failed"] = str(ex)

    return np.ascontiguousarray(u8), mask, rep


def summarise(rep):
    """One log line, or None if nothing interesting happened."""
    bits = []
    if rep.get("band_note"):
        bits.append(rep["band_note"])
    if rep.get("stretch"):
        bits.append(f"stretched ({rep['stretch']})")
    if rep.get("masked_frac", 0) > 0.001:
        bits.append(f"masked {100 * rep['masked_frac']:.1f}% "
                    f"(cloud {100 * rep.get('cloud_frac', 0):.1f}%, "
                    f"shadow {100 * rep.get('shadow_frac', 0):.1f}%)")
    if rep.get("mask_rejected"):
        bits.append(rep["mask_rejected"])
    if rep.get("clahe"):
        bits.append("CLAHE on luminance")
    return " | ".join(bits) if bits else None

In [ ]:
%%writefile dw/mathsandml/benchmark/fetch_data.py
#!/usr/bin/env python3
"""
fetch_data.py - build benchmark scenes from open LiDAR + orthophoto services.

Each scene is a matched pair: a georeferenced optical image and a reference
elevation surface over the same footprint, in the same CRS, at a comparable
date. Everything here is open data that needs no registration, with one
exception (ISPRS) which is handled as a manual drop-in.

    python fetch_data.py --check                 probe every service, download nothing
    python fetch_data.py --list                  show the built-in areas of interest
    python fetch_data.py --source ahn            fetch the Dutch scenes
    python fetch_data.py --source ahn --aoi rotterdam_centre --size-m 400
    python fetch_data.py --source all --out benchmark/scenes

Layer and coverage names are DISCOVERED from each service's GetCapabilities
rather than hard-coded, because published names drift. If discovery fails the
script says exactly which service and which pattern missed, so the fix is a
one-line change to the SOURCES table below rather than a debugging session.

Sources
-------
ahn        Netherlands. AHN LiDAR DSM + DTM at 0.5 m, and the national 8 cm
           orthophoto. Both EPSG:28992. The cleanest pairing available: true
           LiDAR surface, dense low-rise and high-rise cities, flat farmland.
swisstopo  Switzerland. swissSURFACE3D Raster DSM at 0.5 m + SWISSIMAGE.
           EPSG:2056. This is where the hilly and forested rows come from.
usgs       United States. 3DEP elevation + NAIP imagery. NOTE 3DEP is a BARE
           EARTH model, so a USGS scene scores terrain, not building heights.
           Useful for the hilly row, misleading for the urban one.
isprs      Vaihingen / Potsdam. Requires a registration form, so this source
           only verifies and normalises files you downloaded yourself.
"""

import argparse
import io
import json
import os
import re
import sys
import xml.etree.ElementTree as ET

import numpy as np

try:
    import requests
except ImportError:
    sys.exit("pip install requests")

TIMEOUT = 120
UA = {"User-Agent": "depthwizard-benchmark/1.0"}


# ---------------------------------------------------------------------------
# areas of interest:  name -> (crs, centre_x, centre_y, landscape, note)
# ---------------------------------------------------------------------------
AOIS = {
    "ahn": {
        "rotterdam_centre": ("EPSG:28992", 92600, 436700, "urban",
                             "mid-rise perimeter blocks around a park, 6-10 storeys"),
        "delft_old":        ("EPSG:28992", 84400, 447300, "urban",
                             "dense low-rise, narrow streets"),
        "flevoland_farm":   ("EPSG:28992", 165000, 502000, "sparse",
                             "motorway and canal corridor with deciduous shelter belts"),
        "veluwe_forest":    ("EPSG:28992", 185500, 464000, "forest",
                             "a third conifer plantation, the rest open heath and a pond"),
    },
    "swisstopo": {
        "zurich_centre":    ("EPSG:2056", 2683400, 1247500, "urban",
                             "mid-rise European core on a slope"),
        # named for the region, not the relief: this footprint turned out to
        # be chalets and meadow with only 22 m of relief, so it scores as sparse
        "interlaken_hills": ("EPSG:2056", 2633000, 1170000, "sparse",
                             "detached chalets and meadow on a gentle slope"),
        # the genuinely steep ones - the hilly row of the marking scheme
        "wengen_slope":     ("EPSG:2056", 2637000, 1161750, "hilly",
                             "car-free village on a ~30% mountainside: hotels, "
                             "chalets and forest strips on real relief"),
        "lavaux_terraces":  ("EPSG:2056", 2546950, 1149250, "hilly",
                             "terraced vineyards and a wine village above Lake "
                             "Geneva - steep, stepped, built"),
        "jura_forest":      ("EPSG:2056", 2570000, 1220000, "forest",
                             "mixed forest on rolling terrain"),
    },
    "usgs": {
        "denver_metro":     ("EPSG:4326", -104.9903, 39.7392, "urban",
                             "3DEP is bare earth - terrain only, no buildings"),
        "appalachian_ridge":("EPSG:4326", -82.5515, 35.5951, "hilly",
                             "strong relief with forest cover"),
    },
}

SOURCES = {
    # Verified live against the running services on 2026-09-03:
    #   WCS coverages  -> dsm_05m, dtm_05m   (AHN4, 0.5 m, EPSG:28992 only,
    #                                         GEOTIFF only, >=10 pts/m2)
    #   WMS layers     -> Actueel_orthoHR (8 cm), Actueel_ortho25 (25 cm)
    #   WMS GetMap offers image/jpeg ONLY, and caps size at 2500 x 2500.
    "ahn": dict(
        wcs="https://service.pdok.nl/rws/actueel-hoogtebestand-nederland/wcs/v1_0",
        wms="https://service.pdok.nl/hwh/luchtfotorgb/wms/v1_0",
        # priority order: the sharpest product first, a generic catch last
        dsm_pat=[r"dsm[_\-]?0?5", r"\bdsm\b", r"dsm"],
        dtm_pat=[r"dtm[_\-]?0?5", r"\bdtm\b", r"dtm"],
        rgb_pat=[r"ortho.*(hr|8\s*cm)", r"actueel.*ortho", r"ortho"],
        max_px=2500,
        native_px=0.5,
    ),
    "swisstopo": dict(
        stac="https://data.geo.admin.ch/api/stac/v0.9",
        dsm_collection="ch.swisstopo.swisssurface3d-raster",
        # the matching LiDAR terrain model; with it the reference nDSM is a
        # measured difference instead of a low-pass approximation
        dtm_collection="ch.swisstopo.swissalti3d",
        rgb_collection="ch.swisstopo.swissimage-dop10",
        native_px=0.5,
    ),
    "usgs": dict(
        dem="https://elevation.nationalmap.gov/arcgis/rest/services/3DEPElevation/ImageServer",
        rgb="https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPPlus/ImageServer",
        native_px=1.0,
    ),
}


def get(url, params=None, stream=False):
    r = requests.get(url, params=params, headers=UA, timeout=TIMEOUT, stream=stream)
    r.raise_for_status()
    return r


def strip_ns(tag):
    return tag.split("}")[-1].lower()


def bbox_from_centre(cx, cy, size_m):
    h = size_m / 2.0
    return (cx - h, cy - h, cx + h, cy + h)


def write_geotiff(path, arr, bbox, crs, count=None):
    """Write an array with an explicit bbox, so georeferencing never depends on
    whatever the service chose to embed (WMS often embeds nothing)."""
    import rasterio
    from rasterio.transform import from_bounds
    a = np.asarray(arr)
    if a.ndim == 2:
        a = a[None]
    elif a.shape[-1] in (3, 4):
        a = np.transpose(a[..., :3], (2, 0, 1))
    n = count or a.shape[0]
    w, s, e, nn = bbox
    tr = from_bounds(w, s, e, nn, a.shape[2], a.shape[1])
    with rasterio.open(path, "w", driver="GTiff", height=a.shape[1],
                       width=a.shape[2], count=n, dtype=a.dtype,
                       crs=crs, transform=tr) as ds:
        for i in range(n):
            ds.write(a[i], i + 1)
    return path


# ---------------------------------------------------------------------------
# OGC discovery
# ---------------------------------------------------------------------------
def _best(names, patterns):
    """First name matching the first pattern that matches anything.

    Patterns are a priority list, so the highest-resolution product wins over a
    generic fallback: an 8 cm ortho and a 25 cm ortho both match /ortho/, and
    without an order the choice would come down to XML document order.
    """
    if isinstance(patterns, str):
        patterns = [patterns]
    for pat in patterns:
        hit = next((n for n in names if re.search(pat, n, re.I)), None)
        if hit:
            return hit
    return None


def discover_wcs(base, patterns):
    """Return the coverage name matching patterns, and the full list."""
    x = get(base, dict(SERVICE="WCS", VERSION="1.0.0",
                       REQUEST="GetCapabilities")).content
    root = ET.fromstring(x)
    names = [e.text.strip() for e in root.iter()
             if strip_ns(e.tag) == "name" and e.text and e.text.strip()]
    names = [n for n in names if not n.lower().startswith("wcs")]
    return _best(names, patterns), names


def discover_wms(base, patterns):
    """Return (layer, all layers, GetMap formats).

    The formats matter: PDOK's orthophoto WMS advertises image/jpeg and nothing
    else, so a hard-coded image/png request comes back as a ServiceException.
    Read what the service actually offers instead of assuming.
    """
    x = get(base, dict(SERVICE="WMS", VERSION="1.3.0",
                       REQUEST="GetCapabilities")).content
    root = ET.fromstring(x)
    names = []
    for lay in root.iter():
        if strip_ns(lay.tag) != "layer":
            continue
        for ch in lay:
            if strip_ns(ch.tag) == "name" and ch.text:
                names.append(ch.text.strip())

    formats = []
    for req in root.iter():
        if strip_ns(req.tag) != "getmap":
            continue
        for ch in req.iter():
            if strip_ns(ch.tag) == "format" and ch.text:
                formats.append(ch.text.strip())
    return _best(names, patterns), names, formats


def pick_format(formats):
    """Prefer lossless, accept lossy, and never invent one the service lacks."""
    for want in ("image/png", "image/tiff", "image/jpeg"):
        for f in formats:
            if f.lower().startswith(want):
                return f
    return formats[0] if formats else "image/jpeg"


def wcs_coverage(base, coverage, bbox, crs, width, height, out):
    r = get(base, dict(SERVICE="WCS", VERSION="1.0.0", REQUEST="GetCoverage",
                       COVERAGE=coverage, CRS=crs, RESPONSE_CRS=crs,
                       BBOX=",".join(f"{v:.3f}" for v in bbox),
                       WIDTH=width, HEIGHT=height, FORMAT="GEOTIFF"))
    if b"ServiceException" in r.content[:2000] or b"<?xml" in r.content[:20]:
        raise RuntimeError(f"WCS refused the request: {r.content[:400]!r}")
    with open(out, "wb") as f:
        f.write(r.content)
    return out


def wms_image(base, layer, bbox, crs, width, height, fmt="image/jpeg"):
    from PIL import Image
    r = get(base, {"SERVICE": "WMS", "VERSION": "1.3.0", "REQUEST": "GetMap",
                   "LAYERS": layer, "STYLES": "", "CRS": crs,
                   # WMS 1.3.0 honours the CRS axis order. EPSG:28992 and
                   # EPSG:2056 are both easting-first, so minx,miny,maxx,maxy
                   # is correct here - EPSG:4326 would need lat first.
                   "BBOX": ",".join(f"{v:.3f}" for v in bbox),
                   "WIDTH": width, "HEIGHT": height, "FORMAT": fmt})
    head = r.content[:2000]
    if b"ServiceException" in head or head[:5] == b"<?xml":
        raise RuntimeError(f"WMS refused the request: {r.content[:400]!r}")
    return np.array(Image.open(io.BytesIO(r.content)).convert("RGB"))


# ---------------------------------------------------------------------------
# sources
# ---------------------------------------------------------------------------
def fetch_ahn(aoi, cfg, size_m, px, outdir):
    S = SOURCES["ahn"]
    crs, cx, cy, landscape, note = cfg
    bbox = bbox_from_centre(cx, cy, size_m)
    n = int(round(size_m / px))

    if n > S["max_px"]:
        raise RuntimeError(
            f"{n} px exceeds the service cap of {S['max_px']}. "
            f"Reduce --size-m (max {S['max_px'] * px:.0f} m at {px} m/px) "
            f"or raise --px.")

    dsm_name, all_cov = discover_wcs(S["wcs"], S["dsm_pat"])
    if not dsm_name:
        raise RuntimeError(f"no DSM coverage matched {S['dsm_pat']} in {all_cov}")
    dtm_name, _ = discover_wcs(S["wcs"], S["dtm_pat"])
    rgb_name, all_lay, formats = discover_wms(S["wms"], S["rgb_pat"])
    if not rgb_name:
        raise RuntimeError(f"no ortho layer matched {S['rgb_pat']} in {all_lay[:40]}")
    fmt = pick_format(formats)
    print(f"    coverage={dsm_name}  dtm={dtm_name}  ortho={rgb_name}  fmt={fmt}")

    wcs_coverage(S["wcs"], dsm_name, bbox, crs, n, n,
                 os.path.join(outdir, "ref_dsm.tif"))
    if dtm_name:
        try:
            wcs_coverage(S["wcs"], dtm_name, bbox, crs, n, n,
                         os.path.join(outdir, "ref_dtm.tif"))
        except Exception as e:
            print(f"    (no DTM: {e})")
    rgb = wms_image(S["wms"], rgb_name, bbox, crs, n, n, fmt)
    write_geotiff(os.path.join(outdir, "rgb.tif"), rgb, bbox, crs)
    return dict(landscape=landscape, note=note, source="AHN4 / PDOK",
                crs=crs, bbox=list(bbox), px_size_m=px,
                coverage=dsm_name, ortho=rgb_name, wms_format=fmt)


def fetch_swisstopo(aoi, cfg, size_m, px, outdir, parts=("dsm", "dtm", "rgb")):
    S = SOURCES["swisstopo"]
    crs, cx, cy, landscape, note = cfg
    bbox = bbox_from_centre(cx, cy, size_m)

    # STAC search wants WGS84
    from rasterio.warp import transform_bounds
    wgs = transform_bounds(crs, "EPSG:4326", *bbox, densify_pts=21)

    def tile_hrefs(collection, ext=(".tif", ".tiff")):
        """Every tile covering the AOI, newest revision of each, finest GSD.

        swisstopo publishes these as 1 km2 tiles, so any AOI can straddle two
        or four of them - taking a single asset would leave blank strips. Each
        tile is also published at several resolutions (SWISSIMAGE ships 0.1 m
        and 2.0 m), so the asset is chosen by eo:gsd, not by document order.
        """
        url = f"{S['stac']}/collections/{collection}/items"
        j = get(url, dict(bbox=",".join(f"{v:.6f}" for v in wgs), limit=100)).json()
        feats = j.get("features") or []
        if not feats:
            raise RuntimeError(f"STAC returned no items for {collection} over {wgs}")

        best = {}
        for f in feats:
            cands = [(a.get("eo:gsd", 9e9), a["href"])
                     for a in (f.get("assets") or {}).values()
                     if a.get("href", "").lower().endswith(ext)]
            if not cands:
                continue
            cands.sort()
            fid = f.get("id", "")
            key = re.search(r"(\d{4}-\d{4})", fid)          # the tile's km coords
            key = key.group(1) if key else fid
            when = (f.get("properties") or {}).get("datetime") or ""
            if key not in best or when > best[key][0]:
                best[key] = (when, cands[0][1])
        if not best:
            raise RuntimeError(f"no GeoTIFF asset among {len(feats)} {collection} items")
        print(f"    {collection}: {len(best)} tile(s)")
        return [href for _, href in best.values()]

    import rasterio
    from rasterio.warp import reproject, Resampling
    from rasterio.transform import from_bounds
    n = int(round(size_m / px))

    def crop_mosaic(hrefs, bands):
        """Window each remote COG onto the output grid; no whole-tile download."""
        dst_tr = from_bounds(*bbox, n, n)
        dst = np.full((bands, n, n), np.nan, np.float32)
        for href in hrefs:
            tmp = np.full((bands, n, n), np.nan, np.float32)
            with rasterio.open(href) as ds:
                reproject(rasterio.band(ds, list(range(1, bands + 1))), tmp,
                          dst_transform=dst_tr, dst_crs=crs,
                          src_nodata=ds.nodata, dst_nodata=np.nan,
                          resampling=Resampling.bilinear)
            fill = np.isnan(dst) & np.isfinite(tmp)
            dst[fill] = tmp[fill]
            if np.isfinite(dst).all():
                break
        gaps = float(np.isnan(dst).mean())
        if gaps > 0.001:
            print(f"    warning: {100 * gaps:.1f}% of the AOI had no tile coverage")
        return dst

    if "dsm" in parts:
        dsm = crop_mosaic(tile_hrefs(S["dsm_collection"]), 1)
        write_geotiff(os.path.join(outdir, "ref_dsm.tif"), dsm[0], bbox, crs)
    if "dtm" in parts:
        try:
            dtm = crop_mosaic(tile_hrefs(S["dtm_collection"]), 1)
            write_geotiff(os.path.join(outdir, "ref_dtm.tif"), dtm[0], bbox, crs)
        except Exception as e:
            print(f"    (no DTM: {type(e).__name__}: {str(e)[:120]})")
    if "rgb" in parts:
        rgb = crop_mosaic(tile_hrefs(S["rgb_collection"]), 3)
        rgb = np.nan_to_num(rgb, nan=0.0)
        write_geotiff(os.path.join(outdir, "rgb.tif"),
                      np.clip(rgb, 0, 255).astype(np.uint8).transpose(1, 2, 0),
                      bbox, crs)
    return dict(landscape=landscape, note=note, source="swissSURFACE3D + SWISSIMAGE",
                crs=crs, bbox=list(bbox), px_size_m=px)


def fetch_usgs(aoi, cfg, size_m, px, outdir):
    S = SOURCES["usgs"]
    _, lon, lat, landscape, note = cfg
    # work in Web Mercator so size_m is metres
    from rasterio.warp import transform
    crs = "EPSG:3857"
    xs, ys = transform("EPSG:4326", crs, [lon], [lat])
    bbox = bbox_from_centre(xs[0], ys[0], size_m)
    n = int(round(size_m / px))
    common = dict(bbox=",".join(f"{v:.3f}" for v in bbox),
                  bboxSR=3857, imageSR=3857, size=f"{n},{n}", f="image")

    r = get(S["dem"] + "/exportImage", dict(common, format="tiff"))
    with open(os.path.join(outdir, "ref_dsm.tif"), "wb") as f:
        f.write(r.content)
    r = get(S["rgb"] + "/exportImage", dict(common, format="tiff"))
    with open(os.path.join(outdir, "rgb_raw.tif"), "wb") as f:
        f.write(r.content)

    import rasterio
    with rasterio.open(os.path.join(outdir, "rgb_raw.tif")) as ds:
        a = ds.read()[:3]
    write_geotiff(os.path.join(outdir, "rgb.tif"),
                  np.transpose(a, (1, 2, 0)).astype(np.uint8), bbox, crs)
    os.remove(os.path.join(outdir, "rgb_raw.tif"))
    return dict(landscape=landscape, note=note + " | 3DEP is BARE EARTH",
                source="USGS 3DEP + NAIP", crs=crs, bbox=list(bbox),
                px_size_m=px, bare_earth_reference=True)


FETCHERS = dict(ahn=fetch_ahn, swisstopo=fetch_swisstopo, usgs=fetch_usgs)


# ---------------------------------------------------------------------------
def check():
    """Probe every endpoint. Downloads nothing; prints what is reachable."""
    print("Probing services (no downloads)\n")
    ok = True
    for name, S in SOURCES.items():
        print(f"[{name}]")
        for key in ("wcs", "wms", "stac", "dem", "rgb"):
            if key not in S:
                continue
            url = S[key]
            try:
                if key == "wcs":
                    hit, all_ = discover_wcs(url, S["dsm_pat"])
                    print(f"  {key:5s} ok   DSM coverage -> {hit!r}  "
                          f"({len(all_)} coverages)")
                    if not hit:
                        ok = False
                elif key == "wms":
                    hit, all_, formats = discover_wms(url, S["rgb_pat"])
                    print(f"  {key:5s} ok   ortho layer  -> {hit!r}  "
                          f"({len(all_)} layers, formats={formats} "
                          f"-> using {pick_format(formats)})")
                    if not hit:
                        ok = False
                elif key == "stac":
                    j = get(f"{url}/collections/{S['dsm_collection']}").json()
                    print(f"  {key:5s} ok   collection   -> {j.get('id')!r}")
                else:
                    j = get(url, dict(f="json")).json()
                    print(f"  {key:5s} ok   service      -> "
                          f"{j.get('name') or j.get('serviceDescription', '')[:50]!r}")
            except Exception as e:
                ok = False
                print(f"  {key:5s} FAIL {type(e).__name__}: {str(e)[:130]}")
        print()
    print("All services reachable." if ok else
          "Some services failed. See BENCHMARK.md for the manual download route.")
    return 0 if ok else 1


def main():
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--source", default="ahn",
                    choices=["ahn", "swisstopo", "usgs", "all"])
    ap.add_argument("--aoi", default=None, help="one area of interest by name")
    ap.add_argument("--out", default="benchmark/scenes")
    ap.add_argument("--size-m", type=float, default=300.0,
                    help="square footprint in metres (300 m at 0.5 m = 600 px)")
    ap.add_argument("--px", type=float, default=0.5, help="output GSD in metres")
    ap.add_argument("--force", action="store_true",
                    help="re-download scenes that already exist (hand-entered "
                         "keys in scene.json are kept either way)")
    ap.add_argument("--check", action="store_true")
    ap.add_argument("--list", action="store_true")
    args = ap.parse_args()

    if args.check:
        return check()
    if args.list:
        for src, aois in AOIS.items():
            print(f"\n[{src}]")
            for name, (crs, x, y, cls, note) in aois.items():
                print(f"  {name:20s} {cls:7s} {crs}  {x},{y}   {note}")
        return 0

    sources = list(FETCHERS) if args.source == "all" else [args.source]
    os.makedirs(args.out, exist_ok=True)
    made, failed = [], []

    for src in sources:
        for name, cfg in AOIS[src].items():
            if args.aoi and name != args.aoi:
                continue
            scene = f"{src}_{name}"
            d = os.path.join(args.out, scene)
            have = all(os.path.exists(os.path.join(d, f))
                       for f in ("rgb.tif", "ref_dsm.tif"))
            if have and not args.force:
                # This used to re-download every scene on every run_all.sh AND
                # overwrite scene.json - which silently deleted the
                # known_height_m a person had read off the ortho, so the next
                # benchmark failed every scene with "no known_height_m".
                print(f"\n[{scene}] already present - skipped (--force to refetch)")
                if src == "swisstopo" and not os.path.exists(
                        os.path.join(d, "ref_dtm.tif")):
                    # older swisstopo scenes were built before the terrain
                    # model was fetched; add just that
                    print("    adding the swissALTI3D terrain model")
                    try:
                        fetch_swisstopo(name, cfg, args.size_m, args.px, d,
                                        parts=("dtm",))
                    except Exception as e:
                        print(f"    (no DTM: {type(e).__name__}: {str(e)[:120]})")
                made.append(scene)
                continue
            os.makedirs(d, exist_ok=True)
            print(f"\n[{scene}] {cfg[3]} - {cfg[4]}")
            try:
                info = FETCHERS[src](name, cfg, args.size_m, args.px, d)
                info.update(aoi=name, size_m=args.size_m)
                # keep what a human wrote into an existing scene.json
                old = {}
                sj = os.path.join(d, "scene.json")
                if os.path.exists(sj):
                    try:
                        with open(sj) as f:
                            old = json.load(f)
                    except Exception:
                        old = {}
                info = {**info, **{k: v for k, v in old.items()
                                   if k in ("known_height_m", "known_height_source",
                                            "gcps", "sun_azimuth", "sun_elevation",
                                            "landscape", "note")}}
                with open(sj, "w") as f:
                    json.dump(info, f, indent=2)
                print(f"    -> {d}")
                if info.get("known_height_m"):
                    made.append(scene)
                    continue
                # run_benchmark refuses to invent the scale prior, and it must
                # never come from the reference raster this scene is scored
                # against. So a freshly fetched scene is not runnable until a
                # human looks at the ortho and writes the number down.
                print(f"       ACTION NEEDED: open {os.path.join(d, 'rgb.tif')} "
                      f"and add to scene.json:")
                print(f'         "known_height_m": <height of the tallest '
                      f'structure you can identify, in metres>,')
                print(f'         "known_height_source": "read off the ortho - '
                      f'NOT from the reference"')
                made.append(scene)
            except Exception as e:
                print(f"    FAILED  {type(e).__name__}: {str(e)[:200]}")
                failed.append((scene, f"{type(e).__name__}: {e}"))

    print(f"\n{len(made)} scene(s) built: {', '.join(made) if made else '-'}")
    if failed:
        print(f"{len(failed)} failed:")
        for s, e in failed:
            print(f"  {s}: {e[:160]}")
        print("\nIf a service name has drifted, run --check to see what it does "
              "publish, then adjust the pattern in SOURCES. Or use the manual "
              "download route in BENCHMARK.md.")
    return 0 if made else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile dw/training/data.py
"""
data.py - training tiles on disk, and the Dataset that feeds HeightNet.

TILE FORMAT (written by prepare_gamus.py and prepare_lidar.py)
    <root>/index.json            {"format": "depthwizard-tiles-v1", "tiles": [...]}
    <root>/<split>/<id>_rgb.jpg  8-bit RGB
    <root>/<split>/<id>_h.png    16-bit, height above ground in CENTIMETRES,
                                 65535 = no data
    <root>/<split>/<id>_cls.png  8-bit class index (GAMUS order), 255 = none
                                 (optional - LiDAR tiles have no labels)
Each tile record: id, split, rgb, height, cls (or null), gsd (m/px), source,
region, landscape.

WHAT THE TRAINING DATASET DOES TO EACH SAMPLE
    1. pick a tile, weighted by source (--mix) so GAMUS cannot drown the
       forest and mountain tiles
    2. pick a ground sample distance log-uniformly in [0.30, 1.20] m and
       resample the tile to it - the model sees every GSD and is TOLD which
    3. random 518 px crop, random 90-degree rotation and flip
    4. photometric: brightness, contrast, saturation, gamma, haze, blur,
       sensor noise, JPEG, occasional greyscale - aerial orthophotos made to
       look like the satellite images the app will actually get
    5. half the time, the pipeline's own conditioning (preprocess.condition:
       band stretch + CLAHE) so the model is at home with what inference
       hands it
"""

import json
import math
import os
import random

import cv2
import numpy as np
import torch

import heightnet as HN

NODATA_CM = 65535
cv2.setNumThreads(0)


# ----------------------------------------------------------------------------
# reading / writing tiles
# ----------------------------------------------------------------------------
def write_tile(root, split, tid, rgb, height_m, cls=None, jpeg_q=95):
    """rgb uint8 HxWx3, height_m float (nan = no data), cls uint8 or None."""
    d = os.path.join(root, split)
    os.makedirs(d, exist_ok=True)
    rp = os.path.join(split, f"{tid}_rgb.jpg")
    hp = os.path.join(split, f"{tid}_h.png")
    cv2.imwrite(os.path.join(root, rp), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, int(jpeg_q)])
    h = np.where(np.isfinite(height_m), np.clip(height_m, 0, 655.0) * 100.0, NODATA_CM)
    cv2.imwrite(os.path.join(root, hp), np.round(h).astype(np.uint16),
                [cv2.IMWRITE_PNG_COMPRESSION, 6])
    cp = None
    if cls is not None:
        cp = os.path.join(split, f"{tid}_cls.png")
        cv2.imwrite(os.path.join(root, cp), cls.astype(np.uint8))
    return rp, hp, cp


def read_rgb(path):
    a = cv2.imread(path, cv2.IMREAD_COLOR)
    if a is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB)


def read_height(path):
    a = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if a is None:
        raise IOError(f"cannot read {path}")
    h = a.astype(np.float32) / 100.0
    h[a == NODATA_CM] = np.nan
    return h


def read_cls(path, shape):
    if not path:
        return np.full(shape, HN.IGNORE_INDEX, np.uint8)
    a = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    return a if a is not None else np.full(shape, HN.IGNORE_INDEX, np.uint8)


def load_index(root):
    with open(os.path.join(root, "index.json")) as f:
        idx = json.load(f)
    tiles = []
    for t in idx["tiles"]:
        t = dict(t)
        t["root"] = root
        tiles.append(t)
    return tiles


def save_index(root, tiles, **extra):
    os.makedirs(root, exist_ok=True)
    clean = [{k: v for k, v in t.items() if k != "root"} for t in tiles]
    tmp = os.path.join(root, "index.json.tmp")
    with open(tmp, "w") as f:
        json.dump(dict(format="depthwizard-tiles-v1", tiles=clean, **extra), f)
    os.replace(tmp, os.path.join(root, "index.json"))


def gather(roots, splits):
    """All tiles from several data roots whose split is in `splits`."""
    out = []
    for r in roots:
        out += [t for t in load_index(r) if t["split"] in splits]
    return out


# ----------------------------------------------------------------------------
# augmentation
# ----------------------------------------------------------------------------
def resize_height(h, size):
    """Nodata-aware area/linear resize of a float height map to (w, h) size."""
    valid = np.isfinite(h).astype(np.float32)
    shrink = size[0] < h.shape[1]
    interp = cv2.INTER_AREA if shrink else cv2.INTER_LINEAR
    a = cv2.resize(np.nan_to_num(h) * valid, size, interpolation=interp)
    b = cv2.resize(valid, size, interpolation=interp)
    out = a / np.maximum(b, 1e-6)
    out[b < 0.5] = np.nan
    return out.astype(np.float32)


def photometric(img, rng):
    x = img.astype(np.float32)
    # brightness / contrast
    a = rng.uniform(0.75, 1.25)
    b = rng.uniform(-20, 20)
    x = x * a + b + (x.mean() * (1 - a)) * rng.uniform(0, 1)
    # saturation / hue
    if rng.random() < 0.8:
        hsv = cv2.cvtColor(np.clip(x, 0, 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 1] *= rng.uniform(0.6, 1.3)
        hsv[..., 0] = (hsv[..., 0] + rng.uniform(-6, 6)) % 180
        x = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)
    # gamma
    if rng.random() < 0.5:
        g = rng.uniform(0.8, 1.25)
        x = 255.0 * (np.clip(x, 0, 255) / 255.0) ** g
    # haze (atmosphere seen from orbit)
    if rng.random() < 0.25:
        t = rng.uniform(0.05, 0.35)
        x = x * (1 - t) + t * rng.uniform(170, 235)
    # blur (lower effective resolution than the GSD claims)
    if rng.random() < 0.35:
        s = rng.uniform(0.3, 1.3)
        x = cv2.GaussianBlur(x, (0, 0), s)
    # sensor noise
    if rng.random() < 0.3:
        x = x + rng.normal(0, rng.uniform(1, 6), x.shape)
    x = np.clip(x, 0, 255).astype(np.uint8)
    if rng.random() < 0.05:                         # panchromatic
        g = cv2.cvtColor(x, cv2.COLOR_RGB2GRAY)
        x = np.repeat(g[..., None], 3, axis=2)
    if rng.random() < 0.3:                          # compression
        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(x, cv2.COLOR_RGB2BGR),
                               [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(40, 91))])
        if ok:
            x = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    return x


def _condition(img):
    try:
        import preprocess as PRE
        out, _, _ = PRE.condition(img)
        return out
    except Exception:
        return img


def parse_mix(s):
    """'gamus=0.65,ahn=0.15,swisstopo=0.2' -> dict."""
    if not s:
        return {}
    out = {}
    for part in s.split(","):
        k, v = part.split("=")
        out[k.strip()] = float(v)
    return out


# ----------------------------------------------------------------------------
# datasets
# ----------------------------------------------------------------------------
class TrainTiles(torch.utils.data.Dataset):
    """Random crops, random GSD, heavy augmentation. Length = samples per 'epoch'."""

    def __init__(self, tiles, length=10000, crop=HN.TILE, gsd_range=(HN.GSD_MIN, HN.GSD_MAX),
                 mix=None, condition_p=0.5, max_upsample=1.5, seed=0):
        if not tiles:
            raise ValueError("no training tiles")
        self.tiles, self.length, self.crop = tiles, int(length), int(crop)
        self.gsd_range, self.condition_p = gsd_range, condition_p
        self.max_upsample, self.seed = max_upsample, seed
        # per-tile sampling weight: the source share from `mix`, split evenly
        # across that source's tiles (and across landscapes inside a source)
        mix = mix or {}
        by_src = {}
        for i, t in enumerate(tiles):
            by_src.setdefault(t["source"], []).append(i)
        w = np.zeros(len(tiles))
        for src, ids in by_src.items():
            share = mix.get(src, 1.0 / len(by_src))
            by_land = {}
            for i in ids:
                by_land.setdefault(tiles[i].get("landscape") or "any", []).append(i)
            for land_ids in by_land.values():
                w[land_ids] = share / len(by_land) / len(land_ids)
        self.p = w / w.sum()
        self.sources = sorted(by_src)

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        wi = torch.utils.data.get_worker_info()
        rng = np.random.default_rng([self.seed, i, wi.id if wi else 0,
                                     int(torch.initial_seed() % (2 ** 31))])
        for _ in range(5):
            t = self.tiles[rng.choice(len(self.tiles), p=self.p)]
            try:
                return self._sample(t, rng)
            except Exception as e:          # a corrupt tile must not kill a 10 h run
                print(f"[data] skipping {t.get('id')}: {e}")
        raise RuntimeError("five unreadable tiles in a row - check the data")

    def _sample(self, t, rng):
        root = t["root"]
        rgb = read_rgb(os.path.join(root, t["rgb"]))
        h = read_height(os.path.join(root, t["height"]))
        cls = read_cls(os.path.join(root, t["cls"]) if t.get("cls") else None, h.shape)
        native = float(t["gsd"])

        lo = max(self.gsd_range[0], native / self.max_upsample)
        hi = max(lo, self.gsd_range[1])
        gsd = float(math.exp(rng.uniform(math.log(lo), math.log(hi))))
        s = native / gsd
        if abs(s - 1) > 0.02:
            size = (max(8, int(round(rgb.shape[1] * s))), max(8, int(round(rgb.shape[0] * s))))
            rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
            h = resize_height(h, size)
            cls = cv2.resize(cls, size, interpolation=cv2.INTER_NEAREST)

        c = self.crop
        H, W = h.shape
        if H < c or W < c:
            ph, pw = max(0, c - H), max(0, c - W)
            rgb = np.pad(rgb, ((0, ph), (0, pw), (0, 0)), mode="reflect")
            h = np.pad(h, ((0, ph), (0, pw)), constant_values=np.nan)
            cls = np.pad(cls, ((0, ph), (0, pw)), constant_values=HN.IGNORE_INDEX)
            H, W = h.shape
        y = int(rng.integers(0, H - c + 1))
        x = int(rng.integers(0, W - c + 1))
        rgb, h, cls = rgb[y:y + c, x:x + c], h[y:y + c, x:x + c], cls[y:y + c, x:x + c]

        k = int(rng.integers(0, 4))
        rgb, h, cls = np.rot90(rgb, k), np.rot90(h, k), np.rot90(cls, k)
        if rng.random() < 0.5:
            rgb, h, cls = rgb[:, ::-1], h[:, ::-1], cls[:, ::-1]
        rgb = photometric(np.ascontiguousarray(rgb), rng)
        if rng.random() < self.condition_p:
            rgb = _condition(rgb)
        return dict(image=HN.normalise(rgb)[0],
                    height=torch.from_numpy(np.ascontiguousarray(h, np.float32)),
                    cls=torch.from_numpy(np.ascontiguousarray(cls).astype(np.int64)),
                    gsd=torch.tensor(gsd, dtype=torch.float32),
                    source=self.sources.index(t["source"]))


class EvalTiles(torch.utils.data.Dataset):
    """Deterministic centre crops at the tile's own GSD, for validation during training."""

    def __init__(self, tiles, crop=HN.TILE, limit=None, seed=0):
        tiles = list(tiles)
        if limit and len(tiles) > limit:
            r = random.Random(seed)
            tiles = r.sample(tiles, limit)
        self.tiles, self.crop = tiles, crop

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, i):
        t = self.tiles[i]
        root = t["root"]
        rgb = read_rgb(os.path.join(root, t["rgb"]))
        h = read_height(os.path.join(root, t["height"]))
        cls = read_cls(os.path.join(root, t["cls"]) if t.get("cls") else None, h.shape)
        c = self.crop
        H, W = h.shape
        if H < c or W < c:
            ph, pw = max(0, c - H), max(0, c - W)
            rgb = np.pad(rgb, ((0, ph), (0, pw), (0, 0)), mode="reflect")
            h = np.pad(h, ((0, ph), (0, pw)), constant_values=np.nan)
            cls = np.pad(cls, ((0, ph), (0, pw)), constant_values=HN.IGNORE_INDEX)
            H, W = h.shape
        y, x = (H - c) // 2, (W - c) // 2
        return dict(image=HN.normalise(np.ascontiguousarray(rgb[y:y + c, x:x + c]))[0],
                    height=torch.from_numpy(np.ascontiguousarray(h[y:y + c, x:x + c])),
                    cls=torch.from_numpy(cls[y:y + c, x:x + c].astype(np.int64)),
                    gsd=torch.tensor(float(t["gsd"]), dtype=torch.float32),
                    source=0, idx=i)

In [ ]:
%%writefile dw/training/losses.py
"""
losses.py - what HeightNet is trained to minimise.

  ENGINE B (metres):
  total = l1        per-pixel |height - LiDAR|, with LONG-TAIL WEIGHTS
        + w_ce      soft cross-entropy over the height bins
        + w_grad    multi-scale gradient matching (sharp building edges)
        + w_chamfer bin centres pulled toward the heights actually present
        + w_seg     semantic cross-entropy (GAMUS labels; skipped where absent)
  ENGINE A (relative depth - the problem statement's workflow):
        + w_ssi     scale-and-shift-invariant L1 (MiDaS / Depth-Anything's own
                    training loss): prediction and LiDAR are each normalised by
                    median and mean absolute deviation, then compared. Teaches
                    the pretrained relative-depth head to RANK heights correctly
                    from above, while leaving metres to the calibration module.
        + w_ssi_grad  gradient matching on the normalised maps

LONG-TAIL WEIGHTS. Most pixels are ground (0-2 m). A plain mean loss is then
dominated by ground and the network learns to under-shoot tall buildings - the
exact failure of the zero-shot pipeline. Every pixel is weighted by
(frequency of its height band in this batch) ^ -power, normalised to mean 1,
so a 40 m roof pixel counts for more than a lawn pixel. power 0.5 is a
deliberately mild rebalancing (1.0 would be full inverse frequency).

Everything is computed at the network's output resolution (half the input);
targets are area-averaged down to it with nodata respected.
"""

import torch
import torch.nn.functional as F

import heightnet as HN

DEFAULT_WEIGHTS = dict(l1=1.0, ce=0.3, grad=0.5, chamfer=0.1, seg=0.3, ssi=0.5, ssi_grad=0.25)


def downsample_target(h, valid, size):
    """Area-average a (B,H,W) height map with a (B,H,W) bool mask to `size`."""
    v = valid.float()[:, None]
    hs = F.adaptive_avg_pool2d((torch.nan_to_num(h) * valid.float())[:, None], size)
    vs = F.adaptive_avg_pool2d(v, size)
    out = hs / vs.clamp_min(1e-6)
    return out[:, 0], (vs[:, 0] > 0.5)


def downsample_labels(cls, size):
    """Nearest-neighbour subsample of (B,H,W) int labels."""
    return F.interpolate(cls[:, None].float(), size=size, mode="nearest")[:, 0].long()


def longtail_weights(h, valid, n_bands=16, power=0.5, clip=(0.2, 10.0)):
    u = HN.u_of(h)
    umax = float(torch.log1p(torch.tensor(HN.H_MAX / HN.H0)))
    band = (u / umax * n_bands).long().clamp(0, n_bands - 1)
    band_v = band[valid]
    if band_v.numel() == 0:
        return torch.ones_like(h)
    freq = torch.bincount(band_v, minlength=n_bands).float()
    freq = freq / freq.sum()
    wb = (freq.clamp_min(1e-4)) ** (-power)
    w = wb[band]
    w = w / w[valid].mean().clamp_min(1e-6)
    return w.clamp(*clip) * valid.float()


def gradient_loss(pred, target, valid, scales=4):
    """MiDaS-style: penalise |grad(pred - target)| where both pixels are valid."""
    total = pred.new_zeros(())
    for s in range(scales):
        step = 2 ** s
        r = (pred - target)[:, ::step, ::step]
        m = valid[:, ::step, ::step].float()
        gx = (r[:, :, 1:] - r[:, :, :-1]).abs() * m[:, :, 1:] * m[:, :, :-1]
        gy = (r[:, 1:, :] - r[:, :-1, :]).abs() * m[:, 1:, :] * m[:, :-1, :]
        n = m.sum().clamp_min(1.0)
        total = total + (gx.sum() + gy.sum()) / n
    return total / scales


def chamfer_loss(centers, h, valid, n_samples=2048):
    """Bidirectional chamfer (in log-height space) between each image's bin
    centres and a sample of its true heights - AdaBins' bin-placement loss."""
    uc = HN.u_of(centers)                        # (B,K)
    losses = []
    for b in range(h.shape[0]):
        hv = h[b][valid[b]]
        if hv.numel() < 16:
            continue
        if hv.numel() > n_samples:
            idx = torch.randint(0, hv.numel(), (n_samples,), device=hv.device)
            hv = hv[idx]
        ut = HN.u_of(hv)
        d = (ut[:, None] - uc[b][None]) ** 2      # (N,K)
        losses.append(d.min(1).values.mean() + d.min(0).values.mean())
    if not losses:
        return centers.new_zeros(())
    return torch.stack(losses).mean()


def ssi_loss(rel, h, valid, min_mad_m=0.3):
    """Scale-and-shift-invariant L1 + gradient matching, per image. Flat tiles
    (nothing taller than ~min_mad_m) carry no ranking to learn and are skipped."""
    l_abs, l_grad = [], []
    for b in range(rel.shape[0]):
        v = valid[b]
        if int(v.sum()) < 256:
            continue
        t = h[b][v]
        tm = t.median()
        ts = (t - tm).abs().mean()
        if float(ts) < min_mad_m:
            continue
        p = rel[b].float()
        pv = p[v]
        pm = pv.median()
        ps = (pv - pm).abs().mean().clamp_min(1e-4)
        pn = (p - pm) / ps
        tn = (torch.nan_to_num(h[b]) - tm) / ts
        l_abs.append((pn - tn).abs()[v].mean())
        l_grad.append(gradient_loss(pn[None], torch.where(v, tn, torch.zeros_like(tn))[None],
                                    v[None]))
    if not l_abs:
        z = rel.float().sum() * 0.0          # keeps the graph (DDP) on flat batches
        return z, z
    return torch.stack(l_abs).mean(), torch.stack(l_grad).mean()


def compute_losses(out, batch, weights=None, longtail_power=0.5):
    """out: HeightNet.forward() dict. batch: dict(height (B,H,W) nan=nodata,
    cls (B,H,W) long with 255 = unlabelled). Returns (total, parts dict of floats)."""
    w = dict(DEFAULT_WEIGHTS, **(weights or {}))
    pred = out["height"].float()
    size = pred.shape[-2:]
    h_full = batch["height"]
    valid_full = torch.isfinite(h_full)
    th, tv = downsample_target(h_full, valid_full, size)

    parts = {}
    pw = longtail_weights(th, tv, power=longtail_power)
    denom = pw.sum().clamp_min(1.0)
    l1 = ((pred - th).abs() * pw).sum() / denom
    parts["l1"] = l1

    # soft labels over bins: a Gaussian in log-height space, one base bin wide
    logits = out["logits"].float()
    uc = HN.u_of(out["centers"].float())[:, :, None, None]         # (B,K,1,1)
    ut = HN.u_of(th)[:, None]                                        # (B,1,h,w)
    du = float(torch.log1p(torch.tensor(HN.H_MAX / HN.H0))) / logits.shape[1]
    q = torch.softmax(-((ut - uc) ** 2) / (2 * du * du), dim=1)
    ce_map = -(q * torch.log_softmax(logits, dim=1)).sum(1)
    parts["ce"] = (ce_map * pw).sum() / denom

    parts["grad"] = gradient_loss(pred, torch.where(tv, th, torch.zeros_like(th)), tv)
    parts["chamfer"] = chamfer_loss(out["centers"].float(), th, tv)

    if "rel" in out and (w.get("ssi", 0) > 0 or w.get("ssi_grad", 0) > 0):
        parts["ssi"], parts["ssi_grad"] = ssi_loss(out["rel"], th, tv)

    cls = batch.get("cls")
    if cls is not None and w.get("seg", 0) > 0:
        tc = downsample_labels(cls, size)
        tc = torch.where(tc == 0, torch.full_like(tc, HN.IGNORE_INDEX), tc)   # "others"
        if (tc != HN.IGNORE_INDEX).any():
            parts["seg"] = F.cross_entropy(out["seg"].float(), tc,
                                           ignore_index=HN.IGNORE_INDEX)
        else:
            parts["seg"] = out["seg"].float().sum() * 0.0
    total = sum(w[k] * v for k, v in parts.items() if k in w)
    return total, {k: float(v.detach()) for k, v in parts.items()}

In [ ]:
%%writefile dw/training/train.py
#!/usr/bin/env python3
"""
train.py - fine-tune Depth-Anything-V2 into HeightNet (metres above ground).

    # one GPU
    python train.py --data data/gamus data/lidar --variant small --out runs/small
    # two GPUs (Kaggle T4 x2)
    torchrun --nproc_per_node=2 train.py --data ... --variant small --out runs/small
    # carry on in a new session
    python train.py ... --resume runs/small/state.pt

RECIPE (and where each piece comes from)
  backbone     Depth-Anything-V2 (DINOv2 ViT + DPT), pretrained on 62M images.
               Fine-tuned with a LOW learning rate and layer-wise decay, so the
               early layers keep their general geometry and the late ones adapt
               to the top-down view (Depth Any Canopy, 2024: 5e-6 to 2e-5).
  heads        new; 10-20x the backbone learning rate.
  losses       see losses.py: long-tail-weighted L1 + bin cross-entropy +
               gradient matching + bin chamfer + semantic CE.
  data         GAMUS + AHN/swisstopo, source-balanced (--mix), random GSD
               0.3-1.2 m, satellite-style degradations (data.py).
  EMA          an exponential moving average of the weights is what gets
               validated and shipped - steadier than the raw last step.
  AMP          fp16 autocast + GradScaler on CUDA (T4 / P100 have no bf16).

It writes to --out:
  heightnet_best.pt    the EMA weights with the best validation score (ship this)
  heightnet_last.pt    the latest EMA weights
  state.pt             everything needed to --resume
  log.csv, val.csv     training and validation curves
  samples/             image | LiDAR | prediction | uncertainty, per validation
"""

import argparse
import csv
import datetime
import json
import math
import os
import sys
import time

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "mathsandml"))
import heightnet as HN  # noqa: E402
import data as D  # noqa: E402
import losses as L  # noqa: E402

VARIANT_DEFAULTS = {
    #          batch/gpu  lr_enc   iters
    "small": dict(batch=8, lr_enc=2e-5, iters=30000),
    "base": dict(batch=4, lr_enc=1e-5, iters=24000),
    "large": dict(batch=2, lr_enc=5e-6, iters=20000),
    "tiny": dict(batch=2, lr_enc=1e-4, iters=20),
}


# ----------------------------------------------------------------------------
# distributed helpers
# ----------------------------------------------------------------------------
def setup_dist():
    if int(os.environ.get("WORLD_SIZE", "1")) > 1:
        # rank 0 validates and saves while the others wait at a barrier; the
        # default 10-minute NCCL watchdog is too tight for that on a T4
        dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo",
                                timeout=datetime.timedelta(hours=1))
        rank, world = dist.get_rank(), dist.get_world_size()
        local = int(os.environ.get("LOCAL_RANK", "0"))
        if torch.cuda.is_available():
            torch.cuda.set_device(local)
        return rank, world, local
    return 0, 1, 0


def is_main(rank):
    return rank == 0


def log0(rank, *a):
    if rank == 0:
        print(*a, flush=True)


# ----------------------------------------------------------------------------
# optimiser: layer-wise learning-rate decay on the ViT
# ----------------------------------------------------------------------------
def param_groups(model, lr_enc, lr_dec, lr_head, layer_decay, wd):
    m = model.module if hasattr(model, "module") else model
    n_layers = len(m.da.backbone.encoder.layer)
    groups = {}

    def add(name, p, lr):
        no_wd = p.ndim <= 1 or name.endswith(".bias") or "norm" in name or "token" in name \
            or "position" in name
        key = (lr, no_wd)
        g = groups.setdefault(key, dict(params=[], lr=lr, weight_decay=0.0 if no_wd else wd))
        g["params"].append(p)

    for name, p in m.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("da.backbone."):
            if ".embeddings." in name:
                lid = 0
            elif ".encoder.layer." in name:
                lid = int(name.split(".encoder.layer.")[1].split(".")[0]) + 1
            else:
                lid = n_layers + 1
            add(name, p, lr_enc * layer_decay ** (n_layers + 1 - lid))
        elif name.startswith("da."):
            add(name, p, lr_dec)
        else:
            add(name, p, lr_head)
    out = list(groups.values())
    for g in out:
        g["base_lr"] = g["lr"]
    return out


def lr_factor(step, warmup, total, floor=0.01):
    if step < warmup:
        return (step + 1) / warmup
    t = (step - warmup) / max(1, total - warmup)
    return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))


class EMA:
    def __init__(self, model, decay):
        m = model.module if hasattr(model, "module") else model
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in m.state_dict().items()}

    @torch.no_grad()
    def update(self, model, decay=None):
        d = self.decay if decay is None else decay
        m = model.module if hasattr(model, "module") else model
        for k, v in m.state_dict().items():
            s = self.shadow[k]
            if v.is_floating_point():
                s.mul_(d).add_(v.detach().float(), alpha=1 - d)
            else:
                s.copy_(v)

    def copy_to(self, model):
        m = model.module if hasattr(model, "module") else model
        m.load_state_dict({k: v.to(m.state_dict()[k].dtype) for k, v in self.shadow.items()})


# ----------------------------------------------------------------------------
# validation
# ----------------------------------------------------------------------------
@torch.no_grad()
def validate(model, loader, device, amp, max_batches=None):
    model.eval()
    se = ae = n = 0.0
    tall_se = tall_n = tall_bias = 0.0
    sx = sy = sxx = syy = sxy = 0.0
    inter = np.zeros(len(HN.CLASSES))
    union = np.zeros(len(HN.CLASSES))
    a_se = a_n = 0.0          # engine A: relative depth, scale set by B (learned prior)
    rel_r = []                # engine A: how well the relative depth RANKS heights
    shown = None
    for bi, b in enumerate(loader):
        if max_batches and bi >= max_batches:
            break
        x = b["image"].to(device, non_blocking=True)
        g = b["gsd"].to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            out = model(x, g)
        size = b["height"].shape[-2:]
        pred = F.interpolate(out["height"][:, None].float(), size=size, mode="bilinear",
                             align_corners=False)[:, 0].cpu()
        t = b["height"]
        v = torch.isfinite(t)
        d = (pred - torch.nan_to_num(t))[v]
        se += float((d ** 2).sum())
        ae += float(d.abs().sum())
        n += int(v.sum())
        tv, pv = t[v].double(), pred[v].double()
        sx += float(tv.sum()); sy += float(pv.sum())
        sxx += float((tv * tv).sum()); syy += float((pv * pv).sum()); sxy += float((tv * pv).sum())
        tall = v & (torch.nan_to_num(t) >= 10)
        if tall.any():
            dt = (pred - torch.nan_to_num(t))[tall]
            tall_se += float((dt ** 2).sum())
            tall_bias += float(dt.sum())
            tall_n += int(tall.sum())
        cls = b["cls"]
        if (cls < len(HN.CLASSES)).any():
            pc = F.interpolate(out["seg"].float(), size=size, mode="bilinear",
                               align_corners=False).argmax(1).cpu()
            lab = (cls >= 1) & (cls < len(HN.CLASSES))
            for c in range(1, len(HN.CLASSES)):
                pi, ti = (pc == c) & lab, (cls == c)
                inter[c] += float((pi & ti).sum())
                union[c] += float((pi | ti).sum())
        # engine A, as the hybrid uses it: relative depth minus its ground
        # envelope, scaled by quantile-matching to B. Half resolution is enough
        # for a trend line; evaluate.py does it properly on whole tiles.
        th, tvh = L.downsample_target(t, v, out["rel"].shape[-2:])
        relh = out["rel"].float().cpu().numpy()
        bh = out["height"].float().cpu().numpy()
        for k in range(relh.shape[0]):
            vk = tvh[k].numpy()
            if vk.sum() < 256:
                continue
            gk = th[k].numpy()
            if gk[vk].std() > 0.3 and relh[k][vk].std() > 1e-9:
                rel_r.append(float(np.corrcoef(relh[k][vk], gk[vk])[0, 1]))
            if not HN.relative_informative(relh[k]):
                continue
            det = relh[k] - HN.ground_envelope(relh[k], float(b["gsd"][k]) * 2)
            alpha = HN.learned_scale(det, bh[k], vk)
            if alpha is not None:
                ea = (np.maximum(alpha * det, 0) - gk)[vk]
                a_se += float((ea ** 2).sum())
                a_n += int(vk.sum())
        if shown is None:
            shown = (b["image"][:4], t[:4], pred[:4],
                     F.interpolate(out["sigma"][:4, None].float(), size=size,
                                   mode="bilinear", align_corners=False)[:, 0].cpu())
    model.train()
    if n == 0:
        return {}, shown
    cov = sxy / n - (sx / n) * (sy / n)
    vx, vy = sxx / n - (sx / n) ** 2, syy / n - (sy / n) ** 2
    ious = [inter[c] / union[c] for c in range(1, len(HN.CLASSES)) if union[c] > 0]
    m = dict(rmse=math.sqrt(se / n), mae=ae / n,
             r=cov / math.sqrt(max(vx * vy, 1e-12)),
             rmse_tall=math.sqrt(tall_se / tall_n) if tall_n else float("nan"),
             bias_tall=tall_bias / tall_n if tall_n else float("nan"),
             miou=float(np.mean(ious)) if ious else float("nan"), pixels=n,
             rmse_a=math.sqrt(a_se / a_n) if a_n else float("nan"),
             rel_r=float(np.mean(rel_r)) if rel_r else float("nan"))
    return m, shown


def save_samples(path, shown, vmax=40.0):
    import cv2
    img, t, p, s = shown
    rows = []
    for i in range(img.shape[0]):
        rgb = img[i].numpy().transpose(1, 2, 0) * HN.STD + HN.MEAN
        rgb = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

        def cm(a, top):
            a = np.nan_to_num(a.numpy(), nan=0.0)
            return cv2.cvtColor(cv2.applyColorMap((np.clip(a / top, 0, 1) * 255).astype(np.uint8),
                                                  cv2.COLORMAP_VIRIDIS), cv2.COLOR_BGR2RGB)
        top = max(10.0, float(np.nanpercentile(t[i].numpy(), 99.5)) if torch.isfinite(t[i]).any() else vmax)
        rows.append(np.concatenate([rgb, cm(t[i], top), cm(p[i], top), cm(s[i], top / 4)], 1))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, cv2.cvtColor(np.concatenate(rows, 0), cv2.COLOR_RGB2BGR))


# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description="Fine-tune HeightNet")
    ap.add_argument("--data", nargs="+", required=True, help="tile roots (index.json)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--variant", default="small", choices=list(VARIANT_DEFAULTS))
    ap.add_argument("--iters", type=int, default=None)
    ap.add_argument("--batch", type=int, default=None, help="per GPU")
    ap.add_argument("--accum", type=int, default=1)
    ap.add_argument("--lr-enc", type=float, default=None)
    ap.add_argument("--lr-dec", type=float, default=1e-4)
    ap.add_argument("--lr-head", type=float, default=3e-4)
    ap.add_argument("--layer-decay", type=float, default=0.85)
    ap.add_argument("--wd", type=float, default=0.01)
    ap.add_argument("--warmup", type=int, default=1000)
    ap.add_argument("--crop", type=int, default=HN.TILE)
    ap.add_argument("--mix", default="gamus=0.6,ahn=0.18,swisstopo=0.22")
    ap.add_argument("--longtail-power", type=float, default=0.5)
    ap.add_argument("--loss-weights", default="", help='JSON, e.g. {"seg":0.5}')
    ap.add_argument("--ema", type=float, default=0.999)
    ap.add_argument("--val-every", type=int, default=2000)
    ap.add_argument("--val-n", type=int, default=300, help="tiles per validation set")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--grad-ckpt", action="store_true", help="less memory, ~25%% slower")
    ap.add_argument("--resume", default=None)
    ap.add_argument("--init", default=None, help="start from a HeightNet checkpoint's weights")
    ap.add_argument("--max-hours", type=float, default=11.3,
                    help="save and stop before a Kaggle session is killed")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--log-every", type=int, default=50)
    a = ap.parse_args()

    dflt = VARIANT_DEFAULTS[a.variant]
    a.iters = a.iters or dflt["iters"]
    a.batch = a.batch or dflt["batch"]
    a.lr_enc = a.lr_enc or dflt["lr_enc"]
    rank, world, local = setup_dist()
    device = torch.device(f"cuda:{local}" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda" and not a.no_amp
    torch.manual_seed(a.seed + rank)
    np.random.seed(a.seed + rank)
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.out, exist_ok=True)
    t_start = time.time()

    # ---- data --------------------------------------------------------------
    train_tiles = D.gather(a.data, {"train"})
    val_sets = {}
    for root in a.data:
        vt = [t for t in D.load_index(root) if t["split"] == "val"]
        if vt:
            name = os.path.basename(os.path.normpath(root))
            val_sets[name] = vt
    by_src = {}
    for t in train_tiles:
        by_src[t["source"]] = by_src.get(t["source"], 0) + 1
    log0(rank, f"[data] train tiles {by_src}; val sets "
               f"{ {k: len(v) for k, v in val_sets.items()} }; mix {a.mix}")
    mix = D.parse_mix(a.mix)
    mix = {k: v for k, v in mix.items() if k in by_src} or None
    steps_per_rank = a.iters * a.batch * a.accum
    ds = D.TrainTiles(train_tiles, length=steps_per_rank + 64, crop=a.crop, mix=mix,
                      seed=a.seed * 1000 + rank)
    loader = torch.utils.data.DataLoader(
        ds, batch_size=a.batch, shuffle=False, num_workers=a.workers, pin_memory=True,
        drop_last=True, persistent_workers=a.workers > 0,
        prefetch_factor=4 if a.workers > 0 else None)
    val_loaders = {}
    if is_main(rank):
        for name, vt in val_sets.items():
            vds = D.EvalTiles(vt, crop=a.crop, limit=a.val_n, seed=a.seed)
            val_loaders[name] = torch.utils.data.DataLoader(
                vds, batch_size=max(1, a.batch), shuffle=False,
                num_workers=min(2, a.workers))

    # ---- model -------------------------------------------------------------
    if a.variant == "tiny":
        model = HN.build(da_config=HN.tiny_da_config())
    else:
        model = HN.build(a.variant, pretrained=True)
    if a.init:
        m0, _ = HN.load_checkpoint(a.init)
        model.load_state_dict(m0.state_dict())
        log0(rank, f"[model] initialised from {a.init}")
    # Parameters the forward pass never touches must not require grad, or DDP
    # (find_unused_parameters=False) stops at step 2: DINOv2's masking token,
    # and the first fusion layer's residual unit (it has no residual input).
    for n_, p in model.named_parameters():
        if "mask_token" in n_ or "fusion_stage.layers.0.residual_layer1" in n_:
            p.requires_grad_(False)
    if a.grad_ckpt:
        try:
            model.da.gradient_checkpointing_enable()
        except Exception as e:
            log0(rank, f"[model] gradient checkpointing unavailable: {e}")
    model.to(device)
    n_par = sum(p.numel() for p in model.parameters()) / 1e6
    log0(rank, f"[model] HeightNet-{a.variant}: {n_par:.1f} M parameters, device {device}, "
               f"world {world}, amp {amp}")
    if world > 1:
        model = torch.nn.parallel.DistributedDataParallel(
            model, device_ids=[local] if device.type == "cuda" else None)

    opt = torch.optim.AdamW(param_groups(model, a.lr_enc, a.lr_dec, a.lr_head,
                                         a.layer_decay, a.wd), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    ema = EMA(model, a.ema)
    step, best = 0, float("inf")
    history = []
    if a.resume and os.path.exists(a.resume):
        st = torch.load(a.resume, map_location="cpu", weights_only=False)
        (model.module if world > 1 else model).load_state_dict(st["model"])
        opt.load_state_dict(st["opt"])
        if st.get("scaler"):
            scaler.load_state_dict(st["scaler"])
        ema.shadow = {k: v.float().to(device) for k, v in st["ema"].items()}
        step, best, history = st["step"], st.get("best", best), st.get("history", [])
        log0(rank, f"[resume] from step {step}, best score {best:.3f}")
        # a new session writes to a new folder: bring the best-so-far along, so
        # heightnet_best.pt exists even if this session never beats it
        prev_best = os.path.join(os.path.dirname(os.path.abspath(a.resume)), "heightnet_best.pt")
        here_best = os.path.join(a.out, "heightnet_best.pt")
        if is_main(rank) and os.path.exists(prev_best) and not os.path.exists(here_best):
            import shutil
            shutil.copy(prev_best, here_best)
        # and do not replay the first session's samples
        ds.seed += step
    weights = json.loads(a.loss_weights) if a.loss_weights else None

    def save_state(tag="state.pt"):
        if not is_main(rank):
            return
        m = model.module if world > 1 else model
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                        ema={k: v.half() for k, v in ema.shadow.items()}, step=step, best=best,
                        history=history, args=vars(a)),
                   os.path.join(a.out, tag + ".tmp"))
        os.replace(os.path.join(a.out, tag + ".tmp"), os.path.join(a.out, tag))

    def export(name, metrics=None):
        m = model.module if world > 1 else model
        live = {k: v.clone() for k, v in m.state_dict().items()}
        ema.copy_to(model)
        info = dict(step=step, variant=a.variant, iters=a.iters,
                    data=" + ".join(sorted(by_src)), val=metrics or {},
                    trained_utc=time.strftime("%Y-%m-%d %H:%M", time.gmtime()))
        HN.save_checkpoint(os.path.join(a.out, name), m, a.variant, extra=info)
        m.load_state_dict(live)

    def run_validation():
        nonlocal best
        m = model.module if world > 1 else model
        live = {k: v.clone() for k, v in m.state_dict().items()}
        ema.copy_to(model)
        row = dict(step=step)
        scores = []
        for name, vl in val_loaders.items():
            met, shown = validate(m, vl, device, amp)
            for k, v in met.items():
                row[f"{name}_{k}"] = v
            if met:
                # both engines count: B's metres, and A's relative depth once
                # B has set its scale - the hybrid ships both
                sc = met["rmse"] if not np.isfinite(met["rmse_a"]) else \
                    0.5 * (met["rmse"] + met["rmse_a"])
                scores.append(sc)
                log0(rank, f"[val] {name:8s} B: RMSE {met['rmse']:.2f} m  MAE {met['mae']:.2f}  "
                           f"r {met['r']:.3f}  tall(>=10 m) RMSE {met['rmse_tall']:.2f} "
                           f"bias {met['bias_tall']:+.2f} | A: RMSE {met['rmse_a']:.2f} m  "
                           f"rank r {met['rel_r']:.3f} | mIoU {met['miou']:.3f}")
            if shown is not None:
                save_samples(os.path.join(a.out, "samples", f"{step:06d}_{name}.jpg"), shown)
        m.load_state_dict(live)
        score = float(np.mean(scores)) if scores else float("nan")
        row["score"] = score
        history.append(row)
        newfile = not os.path.exists(os.path.join(a.out, "val.csv"))
        with open(os.path.join(a.out, "val.csv"), "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            if newfile:
                w.writeheader()
            w.writerow(row)
        if np.isfinite(score) and score < best:
            best = score
            export("heightnet_best.pt", row)
            log0(rank, f"[val] new best {score:.3f} -> heightnet_best.pt")
        export("heightnet_last.pt", row)

    # ---- loop --------------------------------------------------------------
    model.train()
    it = iter(loader)
    t0, seen, start_step = time.time(), 0, step
    logf = open(os.path.join(a.out, "log.csv"), "a", newline="") if is_main(rank) else None
    stop = False
    while step < a.iters and not stop:
        f = lr_factor(step, a.warmup, a.iters)
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * f
        opt.zero_grad(set_to_none=True)
        parts_acc = {}
        for micro in range(a.accum):
            b = next(it)
            x = b["image"].to(device, non_blocking=True)
            tgt = dict(height=b["height"].to(device, non_blocking=True),
                       cls=b["cls"].to(device, non_blocking=True))
            g_ = b["gsd"].to(device)
            sync = (micro == a.accum - 1) or world == 1
            ctx = model.no_sync() if (world > 1 and not sync) else _Null()
            with ctx:
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    out = model(x, g_)
                loss, parts = L.compute_losses(out, tgt, weights, a.longtail_power)
                scaler.scale(loss / a.accum).backward()
            for k, v in parts.items():
                parts_acc[k] = parts_acc.get(k, 0.0) + v / a.accum
            seen += x.shape[0] * world
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        ema.update(model, decay=min(a.ema, (1 + step) / (10 + step)))
        step += 1

        if is_main(rank) and (step % a.log_every == 0 or step == 1):
            el = time.time() - t0
            ips = seen / max(el, 1e-6)
            eta = (a.iters - step) * el / max(step - start_step, 1)
            row = dict(step=step, lr=opt.param_groups[0]["lr"], grad_norm=float(gn),
                       img_per_s=ips, **{k: round(v, 4) for k, v in parts_acc.items()})
            print(f"[{step:6d}/{a.iters}] " + " ".join(
                f"{k} {v:.3f}" for k, v in parts_acc.items()) +
                f"  |g| {float(gn):.2f}  {ips:.1f} img/s  eta {eta / 3600:.1f} h", flush=True)
            if logf:
                w = csv.DictWriter(logf, fieldnames=list(row))
                if logf.tell() == 0:
                    w.writeheader()
                w.writerow(row)
                logf.flush()
        if step % a.val_every == 0 or step == a.iters:
            if is_main(rank):
                run_validation()
                save_state()
            if world > 1:
                dist.barrier()
        # rank 0's clock decides for everyone: ranks that stop on different
        # steps deadlock DDP and lose everything since the last validation
        hours = (time.time() - t_start) / 3600
        flag = torch.tensor(float(hours > a.max_hours), device=device)
        if world > 1:
            dist.broadcast(flag, 0)
        if flag.item() > 0:
            log0(rank, f"[time] {hours:.2f} h used - saving and stopping at step {step}. "
                       f"Resume with --resume {os.path.join(a.out, 'state.pt')}")
            stop = True
    if is_main(rank):
        if step % a.val_every != 0 and step != a.iters:
            run_validation()
        save_state()
        with open(os.path.join(a.out, "summary.json"), "w") as f:
            json.dump(dict(step=step, iters=a.iters, best_score=best,
                           finished=step >= a.iters, hours=(time.time() - t_start) / 3600,
                           args=vars(a), last_val=history[-1] if history else None), f, indent=1)
        print(f"[done] step {step}/{a.iters}, best val score {best:.3f} m -> "
              f"{os.path.join(a.out, 'heightnet_best.pt')}", flush=True)
    if world > 1:
        dist.destroy_process_group()


class _Null:
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


if __name__ == "__main__":
    main()

In [ ]:
%%writefile dw/training/evaluate.py
#!/usr/bin/env python3
"""
evaluate.py - score a HeightNet checkpoint on whole held-out tiles, through
the SAME predictor the app uses (tiling, rotations, GSD handling), for all
three ways the app can use it:

    A    relative depth -> ground envelope -> scaled by the learned prior
         (the problem statement's workflow: relative depth calibrated to metres)
    B    the metric head's metres, directly
    A+B  the two fused per pixel by uncertainty - what the app ships

    python evaluate.py --ckpt runs/small/heightnet_best.pt \
        --data data/gamus data/lidar --split test,val --out runs/small/eval

FUSION CALIBRATION. Before scoring, a first pass over --calib-split tiles
decides HOW A and B are blended, and writes the answer into the checkpoint
(the pipeline reads it from there; --no-calibrate keeps what is stored):
  ivw      inverse-variance: each engine's own sigma, scaled by how honest it
           turned out to be (c = median(|error| / sigma) / 0.6745; 1.0 = honest)
  stacked  one weight on A per band of B's predicted height (0-2, 2-10, 10-30,
           30+ m), fitted by least squares to the LiDAR
Whichever has the lower RMSE on the calibration tiles is stored. Stacked can
always fall back to weight 0 on A, so the shipped blend is never worse than B
alone on those tiles. Calibration tiles are then LEFT OUT of the scoring.

Reports RMSE / MAE / bias / correlation overall, per source, per landscape, per
city/region and per HEIGHT BAND (0-2, 2-10, 10-30, 30+ m) - the last one is
where a model that under-shoots towers shows itself. 'flat-zero' is the
do-nothing baseline (0 m everywhere).

The 9-scene LiDAR benchmark (mathsandml/benchmark) scores the full pipeline
with a DEM: benchmark/compare_engines.py.
"""

import argparse
import json
import math
import os
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "mathsandml"))
import heightnet as HN  # noqa: E402
import data as D  # noqa: E402

BANDS = [(0, 2), (2, 10), (10, 30), (30, 1e9)]
ENGINES = ("A", "B", "A+B")


class Acc:
    def __init__(self):
        self.n = 0
        self.se = self.ae = self.sx = self.sy = self.sxx = self.syy = self.sxy = self.bias = 0.0

    def add(self, p, t):
        p, t = p.astype(np.float64), t.astype(np.float64)
        d = p - t
        self.n += d.size
        self.se += float((d * d).sum())
        self.ae += float(np.abs(d).sum())
        self.bias += float(d.sum())
        self.sx += float(t.sum()); self.sy += float(p.sum())
        self.sxx += float((t * t).sum()); self.syy += float((p * p).sum())
        self.sxy += float((t * p).sum())

    def result(self):
        if not self.n:
            return None
        n = self.n
        cov = self.sxy / n - self.sx / n * self.sy / n
        vx = self.sxx / n - (self.sx / n) ** 2
        vy = self.syy / n - (self.sy / n) ** 2
        return dict(rmse=math.sqrt(self.se / n), mae=self.ae / n, bias=self.bias / n,
                    r=cov / math.sqrt(vx * vy) if vx > 0 and vy > 0 else float("nan"),
                    pixels=n)


def tiles_for(roots, splits, limit, seed=0, exclude=()):
    """A random (seeded) sample per data root - not the first N, which for
    GAMUS would be one city only (the ids sort by city)."""
    import random
    out = []
    ex = set(exclude)
    for root in roots:
        tt = [t for t in D.load_index(root) if t["split"] in splits and t["id"] not in ex]
        if limit and len(tt) > limit:
            tt = random.Random(seed).sample(tt, limit)
        out += tt
    return out


def calibrate(hp, tiles, rotations, floor_m=0.3, per_tile=20000):
    """Decide the blend (see the module docstring). Returns the fusion dict."""
    rng = np.random.default_rng(0)
    cols = {k: [] for k in ("a", "b", "sa", "sb", "y")}
    for t in tiles:
        rgb = D.read_rgb(os.path.join(t["root"], t["rgb"]))
        h = D.read_height(os.path.join(t["root"], t["height"]))
        r = hp.predict(rgb, t["gsd"], rotations=rotations)
        c = HN.combine(r, t["gsd"])
        if c["a"] is None:
            continue
        idx = np.flatnonzero(np.isfinite(h))
        if idx.size > per_tile:
            idx = rng.choice(idx, per_tile, replace=False)
        for k, arr in (("a", c["a"]), ("b", r["height"]), ("sa", c["a_sigma"]),
                       ("sb", r["sigma"]), ("y", h)):
            cols[k].append(np.asarray(arr, np.float64).ravel()[idx])
    if not cols["y"]:
        return None
    a, b, sa, sb, y = (np.concatenate(cols[k]) for k in ("a", "b", "sa", "sb", "y"))
    ca = max(float(np.median(np.abs(a - y) / np.maximum(sa, floor_m)) / 0.6745), 1e-3)
    cb = max(float(np.median(np.abs(b - y) / np.maximum(sb, floor_m)) / 0.6745), 1e-3)

    # stacked: least-squares weight on A per band of B's predicted height
    edges = list(HN.STACK_BANDS)
    band = np.clip(np.searchsorted(edges, b, side="right") - 1, 0, len(edges) - 1)
    w_a = []
    for k in range(len(edges)):
        m = band == k
        d = b[m] - a[m]
        den = float((d * d).sum())
        w = float(((b[m] - y[m]) * d).sum() / den) if (m.sum() > 500 and den > 0) else 0.0
        w_a.append(min(max(w, 0.0), 1.0))

    def rmse(p):
        return float(np.sqrt(np.mean((p - y) ** 2)))
    base = dict(c_a=ca, c_b=cb, bands=edges, w_a=w_a, calib_tiles=len(cols["y"]))
    scores = {"A": rmse(a), "B": rmse(b),
              "ivw": rmse(HN.fuse(a, sa, b, sb, dict(base, mode="ivw"))[0]),
              "stacked": rmse(HN.fuse(a, sa, b, sb, dict(base, mode="stacked"))[0])}
    mode = "stacked" if scores["stacked"] <= scores["ivw"] else "ivw"
    print("[calib] RMSE on the calibration tiles: " +
          "  ".join(f"{k} {v:.2f}" for k, v in scores.items()) + f"  -> {mode}", flush=True)
    return dict(base, mode=mode, **{f"calib_rmse_{k}": v for k, v in scores.items()})


def main():
    ap = argparse.ArgumentParser(description="Score HeightNet (A, B, A+B) on held-out tiles")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--data", nargs="+", required=True)
    ap.add_argument("--split", default="test,val")
    ap.add_argument("--calib-split", default="val")
    ap.add_argument("--calib-tiles", type=int, default=60, help="per data root")
    ap.add_argument("--no-calibrate", action="store_true")
    ap.add_argument("--out", required=True)
    ap.add_argument("--rotations", type=int, default=4)
    ap.add_argument("--limit", type=int, default=0, help="tiles per data root, 0 = all")
    ap.add_argument("--device", default="auto")
    a = ap.parse_args()

    hp = HN.HeightPredictor(a.ckpt, device=a.device)
    if not a.no_calibrate:
        csplits = set(s.strip() for s in a.calib_split.split(","))
        ct = tiles_for(a.data, csplits, a.calib_tiles, seed=1)
        print(f"[calib] measuring each engine's honesty on {len(ct)} {sorted(csplits)} tiles",
              flush=True)
        fus = calibrate(hp, ct, a.rotations)
        if fus:
            HN.set_fusion(a.ckpt, fus)
            hp.fusion = fus
            print(f"[calib] mode {fus['mode']}; weight on A per B-height band "
                  f"{[round(w, 2) for w in fus['w_a']]} (bands {fus['bands']} m); "
                  f"c_a {fus['c_a']:.2f} c_b {fus['c_b']:.2f} -> written into {a.ckpt}",
                  flush=True)
        else:
            print("[calib] no tile had anything above ground to calibrate on - kept defaults")
        used = [t["id"] for t in ct]
    else:
        used = []

    splits = set(s.strip() for s in a.split.split(","))
    tiles = tiles_for(a.data, splits, a.limit, seed=2, exclude=used)
    if not tiles:
        sys.exit(f"no tiles with split in {splits}")
    print(f"[eval] {hp.describe()} on {len(tiles)} tiles, {a.rotations} rotations, "
          f"device {hp.device}, fusion {hp.fusion or 'uncalibrated'}", flush=True)

    groups = {}

    def acc(engine, key):
        return groups.setdefault(key, {}).setdefault(engine, Acc())

    t0 = time.time()
    alphas, w_bs = [], []
    for i, t in enumerate(tiles):
        rgb = D.read_rgb(os.path.join(t["root"], t["rgb"]))
        h = D.read_height(os.path.join(t["root"], t["height"]))
        r = hp.predict(rgb, t["gsd"], rotations=a.rotations)
        c = HN.combine(r, t["gsd"], fusion=hp.fusion)
        v = np.isfinite(h)
        tv = h[v]
        preds = {"B": r["height"][v], "A+B": c["fused"][v]}
        if c["a"] is not None:
            preds["A"] = c["a"][v]
            alphas.append(c["alpha"])
            w_bs.append(float(np.mean(c["w_b"][v])))
        keys = ["all", f"source/{t['source']}", f"landscape/{t.get('landscape')}",
                f"region/{t['source']}/{t.get('region')}"]
        for k in keys:
            for e, p in preds.items():
                acc(e, k).add(p, tv)
            acc("flat-zero", k).add(np.zeros_like(tv), tv)
        for lo, hi in BANDS:
            m = (tv >= lo) & (tv < hi)
            if m.any():
                k = f"band/{lo:g}-{hi:g} m" if hi < 1e8 else f"band/{lo:g}+ m"
                for e, p in preds.items():
                    acc(e, k).add(p[m], tv[m])
        if (i + 1) % 25 == 0 or i + 1 == len(tiles):
            el = time.time() - t0
            g = groups["all"]
            now = "  ".join(f"{e} {g[e].result()['rmse']:.2f}" for e in ENGINES if e in g)
            print(f"[eval] {i + 1}/{len(tiles)}  RMSE so far: {now} m  ({el / (i + 1):.1f} s/tile)",
                  flush=True)

    res = {k: {e: acc_.result() for e, acc_ in v.items()} for k, v in sorted(groups.items())}
    summary = dict(checkpoint=a.ckpt, model=hp.describe(), splits=sorted(splits),
                   tiles=len(tiles), rotations=a.rotations, fusion=hp.fusion,
                   learned_alpha_median=float(np.median(alphas)) if alphas else None,
                   mean_weight_of_B=float(np.mean(w_bs)) if w_bs else None, results=res)
    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "eval.json"), "w") as f:
        json.dump(summary, f, indent=1)

    def cell(d, e, key="rmse"):
        x = (d.get(e) or {}).get(key) if d.get(e) else None
        return f"{x:.2f}" if x is not None else "-"
    lines = ["# HeightNet evaluation", "",
             f"{hp.describe()} - {len(tiles)} held-out tiles ({', '.join(sorted(splits))}), "
             f"{a.rotations} rotations.", "",
             "A = relative depth calibrated by the learned prior (the problem statement's "
             "workflow); B = metres directly; A+B = fused per pixel (what the app ships).", "",
             "| group | A RMSE | B RMSE | **A+B RMSE** | A+B MAE | A+B r | flat-zero RMSE |",
             "|---|---|---|---|---|---|---|"]
    for k, d in res.items():
        lines.append(f"| {k} | {cell(d, 'A')} | {cell(d, 'B')} | **{cell(d, 'A+B')}** | "
                     f"{cell(d, 'A+B', 'mae')} | {cell(d, 'A+B', 'r')} | {cell(d, 'flat-zero')} |")
    lines += ["", f"Fusion calibration: {hp.fusion or 'none'}. Mean weight given to B: "
                  f"{summary['mean_weight_of_B']}. Calibration tiles are excluded from the "
                  f"scores above."]
    with open(os.path.join(a.out, "eval.md"), "w") as f:
        f.write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile dw/training/prepare_gamus.py
#!/usr/bin/env python3
"""
prepare_gamus.py - GAMUS (Hugging Face: earthflow/GAMUS) -> DepthWizard tiles.

GAMUS (Xiong et al. 2023, TUM / DLR): 1024 x 1024 aerial tiles at 0.33 m from
US cities, each with a LiDAR nDSM (height above ground, from DSM - DTM) and a
6-class land-cover map. On the Hub it is HDF5, one file per modality:

    images/<split>/<CITY>_<r>_<c>_RGB.h5
    heights/<split>/<CITY>_<r>_<c>_AGL.h5
    classes/<split>/<CITY>_<r>_<c>_CLS.h5

The full set is ~80 GB. This script downloads ONE TILE AT A TIME, converts it
to a ~1 MB JPEG + 16-bit PNG + label PNG, and deletes the HDF5 straight away,
so the peak disk use is a few files and the output is ~10x smaller. It is
restartable: tiles already converted are skipped.

    python prepare_gamus.py --out /kaggle/working/data/gamus
    python prepare_gamus.py --out data/gamus --limit 50          # a quick look
    python prepare_gamus.py --out data/gamus --local-dir ./GAMUS # already downloaded

It prints height statistics from the first tiles so a unit problem (centimetres
instead of metres) shows up in the first minute, not after a day of training.
"""

import argparse
import os
import re
import shutil
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "mathsandml"))
import data as D  # noqa: E402

REPO = "earthflow/GAMUS"
GSD = 0.333
PAT = re.compile(r"^images/(?P<split>[^/]+)/(?P<id>.+)_RGB\.h5$")


def read_h5(path):
    """The largest array in the file, whatever the dataset key is called."""
    import h5py
    best = []

    def visit(_, obj):
        if isinstance(obj, h5py.Dataset) and obj.ndim >= 2:
            best.append(obj)
    with h5py.File(path, "r") as f:
        f.visititems(visit)
        if not best:
            raise ValueError(f"no 2-D array in {path}")
        ds = max(best, key=lambda d: d.size)
        return np.asarray(ds[()])


def to_rgb(a):
    a = np.squeeze(a)
    if a.ndim == 3 and a.shape[0] in (3, 4) and a.shape[-1] not in (3, 4):
        a = a.transpose(1, 2, 0)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, axis=2)
    a = a[..., :3]
    if a.dtype != np.uint8:
        a = a.astype(np.float32)
        if np.nanmax(a) <= 1.5:
            a = a * 255.0
        elif np.nanmax(a) > 255:
            lo, hi = np.nanpercentile(a, [1, 99])
            a = (a - lo) / max(hi - lo, 1e-6) * 255.0
        a = np.clip(np.nan_to_num(a), 0, 255).astype(np.uint8)
    return np.ascontiguousarray(a)


def to_height(a, scale=1.0):
    h = np.squeeze(a).astype(np.float32) * float(scale)
    h[~np.isfinite(h)] = np.nan
    h[h < -5] = np.nan                  # nodata fill values
    h = np.where(np.isfinite(h), np.clip(h, 0, 650), np.nan)
    return h


def to_cls(a):
    c = np.squeeze(a)
    c = np.where(np.isfinite(c), c, 255).astype(np.int64)
    c[(c < 0) | (c > 6)] = 255
    return c.astype(np.uint8)


def list_tiles(local_dir=None, repo=REPO, splits=("train", "val", "test")):
    if local_dir:
        files = []
        for root, _, names in os.walk(local_dir):
            for n in names:
                files.append(os.path.relpath(os.path.join(root, n), local_dir).replace(os.sep, "/"))
    else:
        from huggingface_hub import list_repo_files
        files = list_repo_files(repo, repo_type="dataset")
    fs = set(files)
    out = []
    for f in sorted(files):
        m = PAT.match(f)
        if not m or m["split"] not in splits:
            continue
        tid, split = m["id"], m["split"]
        hf = f"heights/{split}/{tid}_AGL.h5"
        cf = f"classes/{split}/{tid}_CLS.h5"
        if hf not in fs:
            continue
        out.append(dict(id=tid, split=split, rgb_file=f, h_file=hf,
                        cls_file=cf if cf in fs else None))
    return out


def fetch(rel, local_dir, tmp, repo):
    if local_dir:
        return os.path.join(local_dir, rel), False
    from huggingface_hub import hf_hub_download
    for attempt in range(6):
        try:
            p = hf_hub_download(repo, rel, repo_type="dataset", local_dir=tmp)
            return p, True
        except Exception as e:
            if attempt == 5:
                raise
            # 429 = the Hub's rate limit for anonymous downloads: back off
            # hard. An HF_TOKEN (free account) raises the limit a lot.
            wait = 10 * 2 ** attempt if "429" in str(e) else 5 * (attempt + 1)
            time.sleep(wait)


def convert(rec, out, local_dir, tmp, repo, scale, quality):
    tid, split = rec["id"], rec["split"]
    city = tid.split("_")[0]
    entry = dict(id=f"gamus_{tid}", split=split, gsd=GSD, source="gamus", region=city,
                 landscape="urban")
    rp = os.path.join(split, f"gamus_{tid}_rgb.jpg")
    hp = os.path.join(split, f"gamus_{tid}_h.png")
    cp = os.path.join(split, f"gamus_{tid}_cls.png")
    if os.path.exists(os.path.join(out, hp)) and os.path.exists(os.path.join(out, rp)):
        entry.update(rgb=rp, height=hp,
                     cls=cp if os.path.exists(os.path.join(out, cp)) else None, cached=True)
        return entry
    paths = []
    try:
        p, rm = fetch(rec["rgb_file"], local_dir, tmp, repo)
        paths.append((p, rm))
        rgb = to_rgb(read_h5(p))
        p, rm = fetch(rec["h_file"], local_dir, tmp, repo)
        paths.append((p, rm))
        h = to_height(read_h5(p), scale)
        cls = None
        if rec["cls_file"]:
            p, rm = fetch(rec["cls_file"], local_dir, tmp, repo)
            paths.append((p, rm))
            cls = to_cls(read_h5(p))
    finally:
        for p, rm in paths:
            if rm and os.path.exists(p):
                os.remove(p)
    if rgb.shape[:2] != h.shape:
        raise ValueError(f"{tid}: rgb {rgb.shape} vs height {h.shape}")
    rp, hp, cp = D.write_tile(out, split, f"gamus_{tid}", rgb, h, cls, jpeg_q=quality)
    v = h[np.isfinite(h)]
    entry.update(rgb=rp, height=hp, cls=cp,
                 h_p50=float(np.percentile(v, 50)) if v.size else None,
                 h_p99=float(np.percentile(v, 99)) if v.size else None,
                 h_max=float(v.max()) if v.size else None,
                 valid=float(np.isfinite(h).mean()))
    return entry


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--out", required=True)
    ap.add_argument("--repo", default=REPO)
    ap.add_argument("--local-dir", default=None,
                    help="read the HDF5 files from here instead of the Hub")
    ap.add_argument("--splits", default="train,val,test")
    ap.add_argument("--limit", type=int, default=0, help="per split, 0 = all")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--height-scale", type=float, default=1.0,
                    help="multiply GAMUS heights by this (1.0 = they are metres)")
    ap.add_argument("--jpeg-quality", type=int, default=95)
    a = ap.parse_args()

    splits = tuple(s.strip() for s in a.splits.split(","))
    recs = list_tiles(a.local_dir, a.repo, splits)
    if a.limit:
        per = {}
        keep = []
        for r in recs:
            per[r["split"]] = per.get(r["split"], 0) + 1
            if per[r["split"]] <= a.limit:
                keep.append(r)
        recs = keep
    counts = {s: sum(r["split"] == s for r in recs) for s in splits}
    print(f"[gamus] {len(recs)} tiles to convert  {counts}", flush=True)
    if not recs:
        sys.exit("no GAMUS tiles found - check --repo / --local-dir")

    os.makedirs(a.out, exist_ok=True)
    tmp = tempfile.mkdtemp(prefix="gamus_")
    entries, failed = [], []
    t0 = time.time()
    try:
        with ThreadPoolExecutor(a.workers) as ex:
            futs = {ex.submit(convert, r, a.out, a.local_dir, tmp, a.repo,
                              a.height_scale, a.jpeg_quality): r for r in recs}
            for n, f in enumerate(as_completed(futs), 1):
                r = futs[f]
                try:
                    entries.append(f.result())
                except Exception as e:
                    failed.append(r["id"])
                    print(f"[gamus] FAILED {r['id']}: {type(e).__name__}: {e}", flush=True)
                if n == 20 or (n <= 20 and n == len(recs)):
                    p99 = [e["h_p99"] for e in entries if e.get("h_p99") is not None]
                    mx = [e["h_max"] for e in entries if e.get("h_max") is not None]
                    if p99:
                        print(f"[gamus] UNIT CHECK over the first {len(p99)} tiles: "
                              f"p99 height median {np.median(p99):.1f}, max {max(mx):.1f}. "
                              f"Metres expected (p99 roughly 10-60).", flush=True)
                        if np.median(p99) > 300:
                            print("[gamus] WARNING: these look like centimetres or "
                                  "millimetres - rerun with --height-scale 0.01 or 0.001",
                                  flush=True)
                if n % 200 == 0 or n == len(recs):
                    rate = n / max(time.time() - t0, 1e-6)
                    print(f"[gamus] {n}/{len(recs)}  {rate:.1f} tiles/s  "
                          f"eta {(len(recs) - n) / max(rate, 1e-6) / 60:.0f} min", flush=True)
                    D.save_index(a.out, entries, dataset="GAMUS", repo=a.repo)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    for e in entries:
        e.pop("cached", None)
    D.save_index(a.out, sorted(entries, key=lambda e: e["id"]), dataset="GAMUS", repo=a.repo)
    by = {s: sum(e["split"] == s for e in entries) for s in splits}
    print(f"[gamus] done: {len(entries)} tiles {by}, {len(failed)} failed, "
          f"{(time.time() - t0) / 60:.1f} min -> {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile dw/training/prepare_lidar.py
#!/usr/bin/env python3
"""
prepare_lidar.py - Dutch (AHN4) and Swiss (swissSURFACE3D) LiDAR tiles for
the landscapes GAMUS does not have: forest, farmland, hills and mountains,
plus European towns.

GAMUS is five US cities. The marking scheme scores urban, sparse, hilly and
forested scenes, and a model that has only ever seen American downtowns has
no reason to get a Swiss mountainside or a Dutch pine plantation right. These
tiles close that gap. Same open services the benchmark uses (fetch_data.py),
so no account and no key:

  Netherlands  AHN4 DSM + DTM at 0.5 m (PDOK WCS) + national orthophoto (WMS)
  Switzerland  swissSURFACE3D raster (DSM) + swissALTI3D (DTM) at 0.5 m +
               SWISSIMAGE, all from the swisstopo STAC API

Target = DSM - DTM = height above ground, the same quantity GAMUS labels.

THE BENCHMARK STAYS UNSEEN. Every sampled tile centre must be at least
--exclude-km (default 3 km) from every benchmark scene in fetch_data.AOIS, and
whole regions are reserved for validation. A model is never trained within
sight of what it is scored on.

    python prepare_lidar.py --out /kaggle/working/data/lidar
    python prepare_lidar.py --out data/lidar --per-region 5     # quick look
    python prepare_lidar.py --list                              # regions only
"""

import argparse
import math
import os
import random
import shutil
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "mathsandml"))
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "mathsandml", "benchmark"))
import data as D  # noqa: E402

# name: (source, centre_x, centre_y, radius_m, landscape, split)
# Coordinates in the national grid: EPSG:28992 (RD New) for NL, EPSG:2056
# (LV95) for CH. Centres are approximate on purpose - tiles are sampled at
# random inside the radius and anything over sea, lake or outside coverage
# is rejected by the quality checks.
REGIONS = {
    # --- Netherlands: AHN4 ------------------------------------------------
    "amsterdam_zuidas":     ("ahn", 120000, 484500, 2000, "urban", "train"),
    "den_haag_centre":      ("ahn", 81000, 455000, 1500, "urban", "train"),
    "utrecht_centre":       ("ahn", 136000, 456000, 2000, "urban", "train"),
    "eindhoven_centre":     ("ahn", 161500, 383000, 2000, "urban", "train"),
    "groningen_centre":     ("ahn", 233500, 582000, 1500, "urban", "val"),
    "hoge_veluwe":          ("ahn", 180000, 452000, 4000, "forest", "train"),
    "utrechtse_heuvelrug":  ("ahn", 157000, 446000, 4000, "forest", "train"),
    "sallandse_heuvelrug":  ("ahn", 232000, 484000, 3000, "forest", "train"),
    "drents_friese_wold":   ("ahn", 219000, 549000, 4000, "forest", "val"),
    "noordoostpolder":      ("ahn", 183000, 523000, 6000, "sparse", "train"),
    "zeeland_farmland":     ("ahn", 50000, 390000, 6000, "sparse", "train"),
    "groningen_farmland":   ("ahn", 242000, 594000, 6000, "sparse", "val"),
    "zuid_limburg_hills":   ("ahn", 189000, 311000, 5000, "hilly", "train"),
    # --- Switzerland: swisstopo --------------------------------------------
    "bern_centre":          ("swisstopo", 2600500, 1199700, 1500, "urban", "train"),
    "basel_centre":         ("swisstopo", 2611300, 1267300, 1500, "urban", "train"),
    "lausanne_centre":      ("swisstopo", 2538200, 1152600, 1500, "urban", "train"),
    "luzern_centre":        ("swisstopo", 2666000, 1211800, 1500, "urban", "train"),
    "zurich_oerlikon":      ("swisstopo", 2683500, 1252800, 1000, "urban", "train"),
    "geneve_centre":        ("swisstopo", 2500300, 1117800, 1500, "urban", "val"),
    "emmental":             ("swisstopo", 2625000, 1203000, 5000, "hilly", "train"),
    "napf_forest":          ("swisstopo", 2638000, 1210000, 4000, "forest", "train"),
    "sihlwald_forest":      ("swisstopo", 2684500, 1234000, 2500, "forest", "train"),
    "ticino_forest":        ("swisstopo", 2715000, 1110000, 5000, "forest", "train"),
    "jura_delemont":        ("swisstopo", 2593000, 1245000, 5000, "forest", "val"),
    "seeland_farmland":     ("swisstopo", 2580000, 1210000, 5000, "sparse", "train"),
    "broye_farmland":       ("swisstopo", 2562000, 1185000, 5000, "sparse", "train"),
    "grindelwald":          ("swisstopo", 2645500, 1164000, 3000, "hilly", "train"),
    "zermatt":              ("swisstopo", 2624500, 1096000, 3000, "hilly", "train"),
    "engelberg":            ("swisstopo", 2674000, 1186000, 3000, "hilly", "train"),
    "davos":                ("swisstopo", 2783500, 1186500, 4000, "hilly", "val"),
}
CRS = {"ahn": "EPSG:28992", "swisstopo": "EPSG:2056"}


def benchmark_centres():
    """Every benchmark scene centre, per source, from fetch_data.AOIS."""
    try:
        import fetch_data as FD
    except ImportError:
        return {}
    out = {}
    for src, aois in FD.AOIS.items():
        for name, (crs, cx, cy, *_rest) in aois.items():
            out.setdefault(src, []).append((name, float(cx), float(cy)))
    return out


def sample_centres(regions, per_region, tile_m, exclude_km, seed=0):
    bench = benchmark_centres()
    rng = random.Random(seed)
    out = []
    for name, (src, cx, cy, rad, land, split) in regions.items():
        got, tries = [], 0
        while len(got) < per_region and tries < per_region * 50:
            tries += 1
            r = rad * math.sqrt(rng.random())
            a = rng.uniform(0, 2 * math.pi)
            x = round((cx + r * math.cos(a)) / 50) * 50
            y = round((cy + r * math.sin(a)) / 50) * 50
            if any(math.hypot(x - bx, y - by) < exclude_km * 1000
                   for _, bx, by in bench.get(src, [])):
                continue
            if any(math.hypot(x - gx, y - gy) < tile_m * 0.8 for gx, gy in got):
                continue
            got.append((x, y))
        for k, (x, y) in enumerate(got):
            out.append(dict(id=f"{src}_{name}_{k:03d}", source=src, region=name,
                            landscape=land, split=split, cx=x, cy=y))
    return out


# ----------------------------------------------------------------------------
# fetching (the benchmark's own, verified service code)
# ----------------------------------------------------------------------------
_AHN_NAMES = {}


def fetch_ahn_tile(cx, cy, size_m, px, outdir):
    import fetch_data as FD
    S = FD.SOURCES["ahn"]
    if not _AHN_NAMES:
        _AHN_NAMES["dsm"], _ = FD.discover_wcs(S["wcs"], S["dsm_pat"])
        _AHN_NAMES["dtm"], _ = FD.discover_wcs(S["wcs"], S["dtm_pat"])
        rgb_name, _, formats = FD.discover_wms(S["wms"], S["rgb_pat"])
        _AHN_NAMES["rgb"], _AHN_NAMES["fmt"] = rgb_name, FD.pick_format(formats)
    crs = CRS["ahn"]
    bbox = FD.bbox_from_centre(cx, cy, size_m)
    n = int(round(size_m / px))
    FD.wcs_coverage(S["wcs"], _AHN_NAMES["dsm"], bbox, crs, n, n,
                    os.path.join(outdir, "ref_dsm.tif"))
    FD.wcs_coverage(S["wcs"], _AHN_NAMES["dtm"], bbox, crs, n, n,
                    os.path.join(outdir, "ref_dtm.tif"))
    rgb = FD.wms_image(S["wms"], _AHN_NAMES["rgb"], bbox, crs, n, n, _AHN_NAMES["fmt"])
    FD.write_geotiff(os.path.join(outdir, "rgb.tif"), rgb, bbox, crs)


def _stac_hrefs(collection, bbox, crs):
    import re
    import fetch_data as FD
    from rasterio.warp import transform_bounds
    wgs = transform_bounds(crs, "EPSG:4326", *bbox, densify_pts=21)
    url = f"{FD.SOURCES['swisstopo']['stac']}/collections/{collection}/items"
    j = FD.get(url, dict(bbox=",".join(f"{v:.6f}" for v in wgs), limit=100)).json()
    best = {}
    for f in j.get("features") or []:
        cands = [(a.get("eo:gsd", 9e9), a["href"]) for a in (f.get("assets") or {}).values()
                 if a.get("href", "").lower().endswith((".tif", ".tiff"))]
        if not cands:
            continue
        cands.sort()
        key = re.search(r"(\d{4}-\d{4})", f.get("id", ""))
        key = key.group(1) if key else f.get("id", "")
        when = (f.get("properties") or {}).get("datetime") or ""
        if key not in best or when > best[key][0]:
            best[key] = (when, cands[0])
    if not best:
        raise RuntimeError(f"STAC: nothing for {collection}")
    return [c for _, c in best.values()]          # [(gsd, href)]


def _mosaic(hrefs, bbox, crs, n, bands, px):
    import rasterio
    from rasterio.transform import from_bounds
    from rasterio.warp import reproject, Resampling
    dst_tr = from_bounds(*bbox, n, n)
    dst = np.full((bands, n, n), np.nan, np.float32)
    for gsd, href in hrefs:
        # read a COG overview near the target GSD instead of the 10 cm original
        # (a 1 km SWISSIMAGE tile is 10000 x 10000 px at full resolution)
        level = -1
        g = float(gsd) if gsd and gsd < 9e8 else px
        while g * 2 ** (level + 2) <= px * 1.01:
            level += 1
        tmp = np.full((bands, n, n), np.nan, np.float32)
        opened = None
        for lv in ([level] if level >= 0 else []) + [None]:
            try:
                opened = rasterio.open(href, overview_level=lv) if lv is not None \
                    else rasterio.open(href)
                break
            except Exception:
                opened = None
        if opened is None:
            continue
        with opened as ds:
            reproject(rasterio.band(ds, list(range(1, bands + 1))), tmp,
                      dst_transform=dst_tr, dst_crs=crs, src_nodata=ds.nodata,
                      dst_nodata=np.nan, resampling=Resampling.bilinear)
        fill = np.isnan(dst) & np.isfinite(tmp)
        dst[fill] = tmp[fill]
    return dst


def fetch_swisstopo_tile(cx, cy, size_m, px, outdir):
    import fetch_data as FD
    S = FD.SOURCES["swisstopo"]
    crs = CRS["swisstopo"]
    bbox = FD.bbox_from_centre(cx, cy, size_m)
    n = int(round(size_m / px))
    dsm = _mosaic(_stac_hrefs(S["dsm_collection"], bbox, crs), bbox, crs, n, 1, px)
    FD.write_geotiff(os.path.join(outdir, "ref_dsm.tif"), dsm[0], bbox, crs)
    dtm = _mosaic(_stac_hrefs(S["dtm_collection"], bbox, crs), bbox, crs, n, 1, px)
    FD.write_geotiff(os.path.join(outdir, "ref_dtm.tif"), dtm[0], bbox, crs)
    rgb = _mosaic(_stac_hrefs(S["rgb_collection"], bbox, crs), bbox, crs, n, 3, px)
    rgb = np.clip(np.nan_to_num(rgb), 0, 255).astype(np.uint8).transpose(1, 2, 0)
    FD.write_geotiff(os.path.join(outdir, "rgb.tif"), rgb, bbox, crs)


# ----------------------------------------------------------------------------
# folder of rgb.tif / ref_dsm.tif / ref_dtm.tif  ->  one training tile
# ----------------------------------------------------------------------------
def _read(path, band_count=None):
    import rasterio
    with rasterio.open(path) as ds:
        a = ds.read().astype(np.float32)
        nod = ds.nodata
    if nod is not None and np.isfinite(nod):
        a[a == nod] = np.nan
    a[np.abs(a) > 1e30] = np.nan
    return a


def fill_nearest(a):
    """Fill NaNs with the nearest valid value (DTM gaps under buildings)."""
    from scipy.ndimage import distance_transform_edt, gaussian_filter
    m = ~np.isfinite(a)
    if not m.any():
        return a
    if m.all():
        return a
    idx = distance_transform_edt(m, return_distances=False, return_indices=True)
    out = a[tuple(idx)]
    sm = gaussian_filter(out, 3)
    out[m] = sm[m]
    return out


def ndsm_from(dsm, dtm):
    """Height above ground. AHN leaves both rasters empty on water, and the DTM
    empty under buildings: water becomes 0 m (true, and exactly the trap the
    zero-shot pipeline fell into), building footprints get the ground filled
    in from the nearest street."""
    water = ~np.isfinite(dsm) & ~np.isfinite(dtm)
    ground = fill_nearest(dtm)
    h = dsm - ground
    h[water] = 0.0
    h[h < -1.5] = np.nan                  # a real DSM is never far below its DTM
    return np.where(np.isfinite(h), np.clip(h, 0, 400), np.nan).astype(np.float32), water


def convert_folder(folder, out, entry, quality=95):
    rgb = _read(os.path.join(folder, "rgb.tif"))
    rgb = np.clip(np.nan_to_num(rgb[:3]), 0, 255).astype(np.uint8).transpose(1, 2, 0)
    dsm = _read(os.path.join(folder, "ref_dsm.tif"))[0]
    dtm = _read(os.path.join(folder, "ref_dtm.tif"))[0]
    h, water = ndsm_from(dsm, dtm)
    # quality gates
    lum = rgb.mean(2)
    blank = float(((lum < 4) | (rgb.min(2) > 250)).mean())
    valid = float(np.isfinite(h).mean())
    wfrac = float(water.mean())
    reason = None
    if blank > 0.15:
        reason = "blank image"          # outside the orthophoto coverage
    elif valid < 0.8:
        reason = "missing height"
    elif wfrac > 0.5:
        reason = "mostly water"
    if reason:
        return None, reason
    rp, hp, cp = D.write_tile(out, entry["split"], entry["id"], rgb, h, None, jpeg_q=quality)
    v = h[np.isfinite(h)]
    e = dict(entry, rgb=rp, height=hp, cls=None, valid=valid, water=wfrac,
             h_p50=float(np.percentile(v, 50)), h_p99=float(np.percentile(v, 99)),
             h_max=float(v.max()))
    return e, None


def work(rec, out, tile_m, px, quality):
    fetchers = {"ahn": fetch_ahn_tile, "swisstopo": fetch_swisstopo_tile}
    hp = os.path.join(out, rec["split"], f"{rec['id']}_h.png")
    if os.path.exists(hp):
        return dict(rec, rgb=os.path.join(rec["split"], f"{rec['id']}_rgb.jpg"),
                    height=os.path.join(rec["split"], f"{rec['id']}_h.png"),
                    cls=None, gsd=px), None
    tmp = tempfile.mkdtemp(prefix="lidar_")
    try:
        for attempt in range(3):
            try:
                fetchers[rec["source"]](rec["cx"], rec["cy"], tile_m, px, tmp)
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(4 * (attempt + 1))
        return convert_folder(tmp, out, dict(rec, gsd=px), quality)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--out", default=None)
    ap.add_argument("--per-region", type=int, default=35)
    ap.add_argument("--tile-m", type=float, default=512.0)
    ap.add_argument("--px", type=float, default=0.5)
    ap.add_argument("--exclude-km", type=float, default=3.0)
    ap.add_argument("--sources", default="ahn,swisstopo")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--jpeg-quality", type=int, default=95)
    ap.add_argument("--list", action="store_true")
    a = ap.parse_args()

    srcs = {s.strip() for s in a.sources.split(",")}
    regions = {k: v for k, v in REGIONS.items() if v[0] in srcs}
    recs = sample_centres(regions, a.per_region, a.tile_m, a.exclude_km, a.seed)
    if a.list or not a.out:
        for name, (src, cx, cy, rad, land, split) in regions.items():
            n = sum(r["region"] == name for r in recs)
            print(f"  {name:22s} {src:9s} {land:7s} {split:5s} {n:3d} tiles  ({cx}, {cy}) r={rad} m")
        print(f"{len(recs)} tile centres; benchmark scenes excluded within {a.exclude_km:g} km")
        return
    os.makedirs(a.out, exist_ok=True)
    print(f"[lidar] {len(recs)} tiles of {a.tile_m:g} m at {a.px:g} m/px from "
          f"{len(regions)} regions", flush=True)
    entries, rejected, failed = [], {}, 0
    t0 = time.time()
    with ThreadPoolExecutor(a.workers) as ex:
        futs = {ex.submit(work, r, a.out, a.tile_m, a.px, a.jpeg_quality): r for r in recs}
        for n, f in enumerate(as_completed(futs), 1):
            r = futs[f]
            try:
                e, why = f.result()
                if e:
                    entries.append(e)
                else:
                    rejected[why] = rejected.get(why, 0) + 1
            except Exception as ex_:
                failed += 1
                print(f"[lidar] FAILED {r['id']}: {type(ex_).__name__}: {str(ex_)[:160]}",
                      flush=True)
            if n % 25 == 0 or n == len(recs):
                rate = n / max(time.time() - t0, 1e-6)
                print(f"[lidar] {n}/{len(recs)}  kept {len(entries)}  rejected "
                      f"{sum(rejected.values())}  failed {failed}  "
                      f"eta {(len(recs) - n) / max(rate, 1e-6) / 60:.0f} min", flush=True)
                D.save_index(a.out, entries, dataset="AHN4 + swisstopo")
    D.save_index(a.out, sorted(entries, key=lambda e: e["id"]), dataset="AHN4 + swisstopo")
    by = {}
    for e in entries:
        k = f"{e['source']}/{e['landscape']}/{e['split']}"
        by[k] = by.get(k, 0) + 1
    print(f"[lidar] done: {len(entries)} tiles kept, rejected {rejected}, failed {failed}")
    for k in sorted(by):
        print(f"    {k:28s} {by[k]}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile dw/training/selftest.py
#!/usr/bin/env python3
"""
selftest.py - the whole training chain on synthetic data, on a CPU, in about
a minute. Run it before spending GPU hours: if this passes, the Kaggle run
fails only for reasons Kaggle can cause (network, quota), not for bugs here.

    python selftest.py

  1. writes fake GAMUS HDF5 files (boxes = buildings, discs = trees, with
     matching heights and labels) in the Hub's folder layout
  2. prepare_gamus.py --local-dir converts them to tiles
  3. train.py --variant tiny runs a few steps (random-weight tiny backbone,
     same code path as the real one), validates, exports a checkpoint
  4. evaluate.py scores that checkpoint through the app's predictor
  5. the checkpoint is reloaded and run on an odd-sized image at an odd GSD
It checks plumbing, not accuracy - 20 steps of a random tiny network learns
nothing useful.
"""

import json
import os
import shutil
import subprocess
import sys
import tempfile

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "mathsandml"))


def fake_scene(rng, n=1024):
    import cv2
    rgb = np.zeros((n, n, 3), np.uint8)
    rgb[:] = (rng.integers(90, 130), rng.integers(110, 150), rng.integers(80, 110))
    h = np.zeros((n, n), np.float32)
    cls = np.ones((n, n), np.uint8)                    # ground
    cv2.rectangle(rgb, (0, n // 2 - 20), (n, n // 2 + 20), (70, 70, 75), -1)
    cls[n // 2 - 20:n // 2 + 20] = 5                   # road
    for _ in range(rng.integers(6, 14)):
        x, y = rng.integers(0, n - 120, 2)
        w, d = rng.integers(30, 120, 2)
        bh = float(rng.choice([4, 8, 15, 30, 60]))
        sh = int(bh * 0.8)
        cv2.rectangle(rgb, (int(x + sh), int(y + sh)), (int(x + w + sh), int(y + d + sh)),
                      (30, 30, 35), -1)
        c = tuple(int(v) for v in rng.integers(120, 220, 3))
        cv2.rectangle(rgb, (int(x), int(y)), (int(x + w), int(y + d)), c, -1)
        h[y:y + d, x:x + w] = bh
        cls[y:y + d, x:x + w] = 3
    for _ in range(rng.integers(5, 15)):
        x, y = rng.integers(20, n - 20, 2)
        r = int(rng.integers(6, 18))
        th = float(rng.uniform(5, 20))
        cv2.circle(rgb, (int(x), int(y)), r, (40, 100, 40), -1)
        yy, xx = np.ogrid[:n, :n]
        m = (yy - y) ** 2 + (xx - x) ** 2 <= r * r
        h[m] = np.maximum(h[m], th)
        cls[m] = 6
    return rgb, h, cls


def write_fake_gamus(root, rng, per_split=dict(train=6, val=2, test=2)):
    import h5py
    for split, k in per_split.items():
        for i in range(k):
            rgb, h, cls = fake_scene(rng)
            tid = f"DC_{i:02d}_{split[:2]}"
            for sub, name, arr in (("images", "RGB", rgb.transpose(2, 0, 1)),
                                   ("heights", "AGL", h),
                                   ("classes", "CLS", cls.astype(np.float32))):
                d = os.path.join(root, sub, split)
                os.makedirs(d, exist_ok=True)
                with h5py.File(os.path.join(d, f"{tid}_{name}.h5"), "w") as f:
                    f.create_dataset("data", data=arr)


def run(cmd):
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=HERE)
    if r.returncode:
        sys.exit(f"FAILED: {' '.join(cmd)}")


def main():
    rng = np.random.default_rng(0)
    work = tempfile.mkdtemp(prefix="hn_selftest_")
    try:
        raw = os.path.join(work, "GAMUS")
        write_fake_gamus(raw, rng)
        tiles = os.path.join(work, "gamus")
        run([sys.executable, "prepare_gamus.py", "--local-dir", raw, "--out", tiles,
             "--workers", "2"])
        idx = json.load(open(os.path.join(tiles, "index.json")))
        assert len(idx["tiles"]) == 10, len(idx["tiles"])
        runs = os.path.join(work, "run")
        run([sys.executable, "train.py", "--data", tiles, "--out", runs, "--variant", "tiny",
             "--iters", "12", "--batch", "2", "--accum", "2", "--val-every", "6",
             "--val-n", "2", "--workers", "0", "--warmup", "3", "--log-every", "3"])
        best = os.path.join(runs, "heightnet_best.pt")
        assert os.path.exists(best) and os.path.exists(os.path.join(runs, "state.pt"))
        run([sys.executable, "train.py", "--data", tiles, "--out", runs, "--variant", "tiny",
             "--iters", "14", "--batch", "2", "--val-every", "7", "--val-n", "2",
             "--workers", "0", "--warmup", "3", "--resume", os.path.join(runs, "state.pt")])
        run([sys.executable, "evaluate.py", "--ckpt", best, "--data", tiles,
             "--split", "test", "--out", os.path.join(runs, "eval"), "--rotations", "2"])
        import heightnet as HN
        hp = HN.HeightPredictor(best, device="cpu")
        print(f"  fusion stored in the checkpoint: {hp.fusion}")
        img = (rng.random((611, 947, 3)) * 255).astype(np.uint8)
        for gsd in (None, 0.25, 0.7, 2.0):
            r = hp.predict(img, gsd, rotations=2)
            assert r["height"].shape == img.shape[:2] and r["seg"].shape == img.shape[:2]
            assert r["rel"].shape == img.shape[:2] and np.isfinite(r["rel"]).all()
            assert np.isfinite(r["height"]).all() and (r["sigma"] >= 0).all()
            c = HN.combine(r, gsd or HN.GSD_DEFAULT, fusion=hp.fusion)
            assert c["fused"].shape == img.shape[:2] and np.isfinite(c["fused"]).all()
            print(f"  predict gsd={gsd}: model gsd {r['gsd_model']:.2f}, {r['tiles']} tiles, "
                  f"B {r['height'].min():.1f}..{r['height'].max():.1f} m, "
                  f"A scale {c['alpha']}, fused mean {c['fused'].mean():.2f} m")
        # A random tiny network has no relative depth to calibrate, so the
        # hybrid path is exercised with a stand-in predictor whose engines
        # have KNOWN errors: B = 0.9 x truth, A = 0.2 x truth + 3 (+ noise).
        import data as D
        import evaluate as E

        class FakePredictor:
            fusion = {}

            def predict(self, rgb, gsd, rotations=4):
                t = np.nan_to_num(self._h)
                n = np.random.default_rng(1).normal(0, 0.3, t.shape)
                return dict(height=0.9 * t + n, sigma=np.full(t.shape, 0.5),
                            rel=0.2 * t + 3.0 + 0.05 * n, rel_sigma=np.full(t.shape, 0.05))
        fp = FakePredictor()
        tl = [t for t in D.load_index(tiles) if t["split"] == "test"]
        for t in tl:
            fp._h = D.read_height(os.path.join(t["root"], t["height"]))
            c = HN.combine(fp.predict(None, t["gsd"]), t["gsd"])
            assert c["alpha"] is not None and 3.5 < c["alpha"] < 5.5, c["alpha"]
        fp._h = D.read_height(os.path.join(tl[0]["root"], tl[0]["height"]))
        fus = E.calibrate(fp, tl[:1], rotations=1)
        assert fus and np.isfinite(fus["c_a"]) and np.isfinite(fus["c_b"]), fus
        assert fus["mode"] in ("ivw", "stacked") and len(fus["w_a"]) == 4, fus
        assert fus["calib_rmse_stacked"] <= fus["calib_rmse_B"] + 1e-9, fus
        print(f"  hybrid on a stand-in with known errors: A scale {c['alpha']:.2f} "
              f"(expect ~4.5), calibration {fus}")
        # the hybrid maths on a scene with a known answer
        det = np.zeros((200, 200)); det[50:100, 50:100] = 3.0; det[120:140, 20:180] = 1.0
        met = 7.5 * det
        a_ = HN.learned_scale(det, met)
        assert abs(a_ - 7.5) < 1e-6, a_
        fh, fs, wb = HN.fuse(np.full(4, 10.0), np.full(4, 1.0), np.full(4, 20.0), np.full(4, 3.0))
        assert abs(fh[0] - 11.0) < 1e-6 and abs(wb[0] - 0.1) < 1e-6, (fh, wb)
        # stacked weights are the least-squares optimum: A = truth + 2, B = truth - 2
        # -> the best blend is 50/50 in every band
        yy = np.linspace(0, 40, 4000)
        fh, _, wb = HN.fuse(yy + 2, np.ones_like(yy), yy - 2, np.ones_like(yy),
                            dict(mode="stacked", w_a=[0.5] * 4))
        assert np.allclose(fh, yy), "stacked blend"
        print("  learned scale and fusion maths: exact")
        print("\nSELFTEST PASSED - data prep, training, resume, export, evaluation and "
              "inference all run.")
    finally:
        shutil.rmtree(work, ignore_errors=True)


if __name__ == "__main__":
    main()

### Sanity check: the whole chain on synthetic data (about a minute)

In [ ]:
sh("cd dw/training && python selftest.py")

### GAMUS -> tiles

In [ ]:
sh(f"python dw/training/prepare_gamus.py --out {OUT}/gamus --workers 16 "
   f"--limit {GAMUS_LIMIT}")

### Dutch and Swiss LiDAR -> tiles

In [ ]:
sh(f"python dw/training/prepare_lidar.py --out {OUT}/lidar --workers 8 "
   f"--per-region {LIDAR_PER_REGION}")

### What came out

In [ ]:
import json, os, collections
for name in ("gamus", "lidar"):
    p = os.path.join(OUT, name, "index.json")
    if not os.path.exists(p):
        print(name, "- missing"); continue
    t = json.load(open(p))["tiles"]
    c = collections.Counter((x["source"], x.get("landscape"), x["split"]) for x in t)
    print(f"{name}: {len(t)} tiles")
    for k, v in sorted(c.items()):
        print("   ", k, v)
!du -sh {OUT}/*
!rm -rf dw     # keep the output to the data only